# MRKR Contralateral TKA — DICOM to contralateral-knee shards (Colab)

Turns the **6,122** transferred pre-index radiographs into one 8-bit **512 x 512 PNG per image
containing the CONTRALATERAL knee and nothing else**, packs them into WebDataset `.tar` shards
grouped by split, and writes a per-image label sidecar (`labels.csv`) next to them on Drive.
`notebooks/train_colab.ipynb` (task T7) reads exactly those shards.

This notebook is the **Colab twin of `src/preprocess_images.py`**. The pipeline cell below is a
verbatim slice of that module, so a crop produced here is byte-identical to a crop produced
locally. If you change one, you must change the other.

## The four success non-negotiables

1. **Contralateral crop fidelity.** Every pre-index film shows **two NATIVE knees** — the index
   TKA has not happened yet — and 4,075 of the 4,269 frontal films are bilateral (`laterality == 'B'`).
   A half-select with the sign inverted silently trains the model on the knee that is about to be
   replaced. Target: **zero index-knee pixels, zero burned-in laterality markers.**
2. **No leakage.** Intensity normalization is strictly **per image** (robust percentile clip +
   min-max). No statistic is pooled across images, so no cross-split quantity exists to leak
   (protocol section 12). Any *learned* localizer must be fit or selected on **training patients only**.
3. **One pre-specified architecture** downstream — this notebook must not create choices for the
   model to be tuned against.
4. **The test split stays SEALED.** `SPLITS` defaults to `train,val`; processing `test` additionally
   requires the explicit `INCLUDE_TEST = True` opt-in, exactly like the module's `--include-test`.

## PREREQUISITE — do not start until this has passed

```
python3 -m src.verify_transfer          # run LOCALLY, on the Mac, before touching this notebook
```

It must print `VERDICT: PASS`. That check confirms all 6,122 manifest paths arrived from Globus,
none is truncated, no unexpected extra DICOM is present, and a random sample opens with real pixel
data. **A `FAIL` there means every number produced downstream is meaningless.** The Drive gate two
in cell 2 is a second, Colab-side check of the *cloud* copy — it does not replace `verify_transfer`.

## Contract fingerprints (drift detector)

| file | sha256 at port time |
| --- | --- |
| `src/preprocess_images.py` | `e3f3bce7571dd59faec3de1f431bc2e668ab1284890525b45e389fcdb0d15dbc` |
| `src/crop_qa.py` | `fe95378b5e94b6ec59e7694e6a24587a7bb131b031674e4d8872e6f54bf4ca13` |
| ported pipeline region | `ac5fea9b4d2e9dd28d14bc1593df7d0c4595dfbc5f6d062840fab7b6b730cc3c` |

Locally: `shasum -a 256 src/preprocess_images.py src/crop_qa.py`. A mismatch means the module moved
and this notebook was not re-ported — **re-port before you trust a shard.**

## Cell order (run top to bottom, nothing is hidden)

0. Mount Drive, set the operator flags
1. Stage the metadata bundle, resolve every Drive path from config
2. **DRIVE GATE** — 6,122 `.dcm` present and set-equal to the manifest
3. **FROZEN pipeline** (verbatim port)  ·  4. Optional learned localizer (disabled by default)
5. Manifest + work list  ·  6. Resumable shard writer  ·  7. Execute  ·  8. Finalize
9. **CROP QA evidence sheet**  ·  10. Shard integrity verification  ·  HANDOFF TO T7

## 0. Mount Drive, resolve paths, set the operator flags

Everything that changes between runs lives in this one cell. Nothing below it takes a hidden
default.

Paths are **derived from `config/feasibility.yaml`**, not hard-coded — including the name of
the metadata folder itself, which is `preprocess.colab_metadata_dir`. The next cell finds the
project folder by looking under `/content/drive/MyDrive` for a `*/*/config/feasibility.yaml`
whose parent folder is named by that key, then takes the DICOM root to be its
`<basename of transfer.dest_root>` child and warns if the folder name has drifted from
`transfer.dest_root`. Shards go to `preprocess.shards.out_dir` (`shards/`) as a **SIBLING of
`DICOMs-knee-imaging`, never inside it** — `src/verify_transfer.py` walks the DICOM root and would
report shards found there as unexpected extra files, failing the transfer gate.

In [ ]:
# ---- 0. Mount Drive + operator flags ---------------------------------------------------------
import os, sys, json, time, shutil, tarfile, hashlib, tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # re-running is a no-op once mounted

# Override DRIVE_ROOT if you are running this in a local Jupyter with Drive mounted elsewhere.
DRIVE_ROOT = Path("/content/drive/MyDrive")

# ---- operator flags (the ONLY knobs in this notebook) ----------------------------------------
SPLITS          = ["train", "val"]   # non-negotiable #4: the sealed test split is NOT here
INCLUDE_TEST    = False              # mirrors `--include-test`; see the loud warning in cell 5
LIMIT           = None               # mirrors `--limit`; None = process everything. Logged loudly.
DRY_RUN         = False              # walk the work list and report; read no pixels
FORCE_RESCAN    = False              # re-walk the Drive DICOM tree instead of using the cache
RESUME          = True               # skip keys already inside a COMPLETED shard
RESUME_STRICT   = False              # verify resume state by reading member names out of every tar
FULL_VERIFY     = False              # cell 10: parse EVERY sample .json instead of a sample
MAX_SHARD_MB_OVERRIDE = None         # None = config value (400). See the note in cell 6.
QA_SEED_OFFSET  = 0                  # draw a different QA sample without touching the config seed

# ---- metadata bundle (see cell 1) --------------------------------------------------------------
# The folder name is NOT set here. It is read back from the bundle's own
# `preprocess.colab_metadata_dir` in the next cell, which is the same key the Mac-side staging
# command derives its DEST from -- so the writer and the reader cannot drift apart silently.

# ---- local scratch (fast, ephemeral) ---------------------------------------------------------
_BASE = Path("/content") if os.path.isdir("/content") else Path(tempfile.gettempdir())
LOCAL_ROOT  = _BASE / "mrkr";                   LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_META  = LOCAL_ROOT / "meta"
LOCAL_STAGE = LOCAL_ROOT / "shard_staging";     LOCAL_STAGE.mkdir(parents=True, exist_ok=True)
CACHE_JSON  = LOCAL_ROOT / "drive_dicom_scan.json"

print("Colab:", IN_COLAB, "| Drive root exists:", DRIVE_ROOT.exists())
print("local scratch:", LOCAL_ROOT)
print("SPLITS =", SPLITS, "| INCLUDE_TEST =", INCLUDE_TEST, "| LIMIT =", LIMIT,
      "| RESUME =", RESUME, "| DRY_RUN =", DRY_RUN)

## 1. Put the metadata bundle on Drive (one-time, do this on the Mac first)

This notebook needs six small, **patient-level** files. They must go to **Drive** (private, the
same place the DICOMs already are) and **never** to a public bucket, a gist, or a git remote.

| file | size | why it is needed |
| --- | --- | --- |
| `derived-data/cohort/selected_study_images.parquet` | 1.01 MB | one row per selected pre-index image: `dicom_path`, `view_position`, `laterality` |
| `derived-data/cohort/final_cohort.parquet` | 285 KB | `index_side` / `contra_side`, `event_indicator`, `time_from_landmark` |
| `derived-data/cohort/patient_splits.parquet` | 37 KB | the LOCKED train / val / test assignment |
| `derived-data/source-parquet/image_flags.parquet` | **233 KB** | the MRKR `horizontal_flip` and `inverted` flags (a slice of the 49 MB `image.parquet`) |
| `config/feasibility.yaml` | 25 KB | every crop parameter; nothing is hard-coded here |
| `outputs/tables/image_transfer_manifest_paths.txt` | 1.02 MB | the 6,122 relative paths the Drive gate checks against |

**Total: about 2.6 MB.** Run this **on the Mac**, from the project root. It copies the bundle into
the Drive project folder **preserving the repo-relative layout**, which is what lets the pipeline
cell use the project's own `Config` loader unmodified:

```bash
# DEST is derived from config: the Drive project folder is the parent of transfer.dest_root
# (override it with $MRKR_DRIVE_ROOT), and the bundle folder is preprocess.colab_metadata_dir.
# Nothing personal is hard-coded, and the notebook reads the same key back.
DEST="$(python3 -c 'import os,yaml; c=yaml.safe_load(open("config/feasibility.yaml")); d=os.path.expandvars(os.environ.get("MRKR_DRIVE_ROOT") or c["transfer"]["dest_root"]); print(os.path.join(os.path.dirname(d), c["preprocess"]["colab_metadata_dir"]))')"
echo "staging the metadata bundle to: $DEST"
mkdir -p "$DEST/derived-data/cohort" "$DEST/derived-data/source-parquet" "$DEST/config" "$DEST/outputs/tables"
cp derived-data/cohort/selected_study_images.parquet "$DEST/derived-data/cohort/"
cp derived-data/cohort/final_cohort.parquet          "$DEST/derived-data/cohort/"
cp derived-data/cohort/patient_splits.parquet        "$DEST/derived-data/cohort/"
cp config/feasibility.yaml                           "$DEST/config/"
cp outputs/tables/image_transfer_manifest_paths.txt  "$DEST/outputs/tables/"
python3 - "$DEST" <<'PY'
import sys, pandas as pd
sel = pd.read_parquet("derived-data/cohort/selected_study_images.parquet", columns=["SOPInstanceUID_anon"])
img = pd.read_parquet("derived-data/source-parquet/image.parquet",
                      columns=["SOPInstanceUID_anon", "horizontal_flip", "inverted"])
sub = img[img["SOPInstanceUID_anon"].isin(set(sel["SOPInstanceUID_anon"]))]
assert sub["SOPInstanceUID_anon"].nunique() == sel["SOPInstanceUID_anon"].nunique()
sub.to_parquet(sys.argv[1] + "/derived-data/source-parquet/image_flags.parquet", index=False)
print("image_flags.parquet:", len(sub), "rows")
PY
```

That last block writes a **233 KB** three-column slice restricted to the 6,578 selected images
instead of shipping the 49 MB `image.parquet`. `build_manifest` reads only those three columns and
left-joins on `SOPInstanceUID_anon`, so the result is identical — and if a UID ever failed to match,
the `MANIFEST_REGRESSION` assertions (`horizontal_flip == 1` on exactly 287, `inverted == 1` on
exactly 1,476) would fail loudly rather than silently substituting a 0. Copying the full
`image.parquet` works too; the staging cell accepts either and prints which one it used.

The bundle folder (`preprocess.colab_metadata_dir`, currently `colab-metadata/`) is a
**sibling** of `DICOMs-knee-imaging/`, so it is invisible to `src/verify_transfer.py`.

In [ ]:
# ---- 1. Resolve the Drive paths from config, stage the metadata bundle locally ----------------
import yaml

# The bundle folder NAME is not hard-coded: every candidate
# <project>/<dir>/config/feasibility.yaml under the Drive root is read, and the one whose
# folder is named by its own `preprocess.colab_metadata_dir` wins. That is the same key the
# Mac-side staging command in the markdown above derives DEST from.
META_DRIVE = DRIVE_PROJECT = META_DIRNAME = None
_tried = []
for _cand in sorted(DRIVE_ROOT.glob("*/*/config/feasibility.yaml")):
    _name = str(yaml.safe_load(_cand.read_text())
                .get("preprocess", {}).get("colab_metadata_dir", ""))
    _tried.append(f"{_cand.parent.parent.name!r} (its config names {_name!r})")
    if _name and _cand.parent.parent.name == _name:
        META_DRIVE = _cand.parent.parent
        DRIVE_PROJECT = META_DRIVE.parent
        META_DIRNAME = _name
        break
assert META_DRIVE is not None, (
    f"metadata bundle not found under {DRIVE_ROOT}. Expected <project folder>/<the folder named "
    f"by preprocess.colab_metadata_dir>/config/feasibility.yaml. Candidates seen: "
    f"{_tried or 'none'}. Run the bash block in the markdown cell above ON THE MAC, wait for "
    f"Google Drive for Desktop to finish uploading it, then re-run this cell.")

# Stage the bundle on local disk: the parquet reads become local instead of Drive-FUSE, and the
# same files are re-read by the QA and verification cells.
if LOCAL_META.exists():
    shutil.rmtree(LOCAL_META)
shutil.copytree(META_DRIVE, LOCAL_META)

flags_full = LOCAL_META / "derived-data" / "source-parquet" / "image.parquet"
flags_slim = LOCAL_META / "derived-data" / "source-parquet" / "image_flags.parquet"
if not flags_full.exists():
    assert flags_slim.exists(), (
        "neither image.parquet nor image_flags.parquet is in the bundle; build_manifest needs "
        "SOPInstanceUID_anon / horizontal_flip / inverted")
    shutil.copy2(flags_slim, flags_full)      # build_manifest asks for image.parquet by name
    FLAGS_SOURCE = "image_flags.parquet (3-column slice)"
else:
    FLAGS_SOURCE = "image.parquet (full)"

CFG_PATH = LOCAL_META / "config" / "feasibility.yaml"
_raw = yaml.safe_load(CFG_PATH.read_text())
assert str(_raw["preprocess"]["colab_metadata_dir"]) == META_DIRNAME == META_DRIVE.name, (
    f"the staged bundle is in {META_DRIVE.name!r} but its config says "
    f"preprocess.colab_metadata_dir = {_raw['preprocess']['colab_metadata_dir']!r}; re-run the staging command")

# Every Drive path is DERIVED from config -- nothing about the folder layout is hard-coded here.
_dest_root   = Path(str(_raw["transfer"]["dest_root"]))
DICOM_ROOT   = DRIVE_PROJECT / _dest_root.name                       # .../DICOMs-knee-imaging
SHARD_DIR    = DRIVE_PROJECT / str(_raw["preprocess"]["shards"]["out_dir"])   # SIBLING, never inside
PROGRESS_DIR = SHARD_DIR / "_progress"
QA_DIR       = DRIVE_PROJECT / "qa"
LOG_DIR      = DRIVE_PROJECT / "logs"
MANIFEST_TXT = LOCAL_META / str(_raw["transfer"]["manifest_paths"])
for d in (SHARD_DIR, PROGRESS_DIR, QA_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert DICOM_ROOT.exists(), f"DICOM root not on Drive: {DICOM_ROOT}"
assert MANIFEST_TXT.exists(), f"manifest path list missing from the bundle: {MANIFEST_TXT}"
assert SHARD_DIR.resolve() != DICOM_ROOT.resolve() and DICOM_ROOT.resolve() not in SHARD_DIR.resolve().parents, (
    f"shards would be written INSIDE the DICOM root ({DICOM_ROOT}); src/verify_transfer.py would "
    f"then report them as unexpected extra files and FAIL the transfer gate")

if DRIVE_PROJECT.name != _dest_root.parent.name:
    print(f"NOTE: the Drive folder is named {DRIVE_PROJECT.name!r} but config transfer.dest_root "
          f"says {_dest_root.parent.name!r}. Harmless if you renamed it on Drive; check it is the "
          f"right folder.")

print("Drive project :", DRIVE_PROJECT)
print("DICOM root    :", DICOM_ROOT)
print("shard out dir :", SHARD_DIR, "  (sibling of the DICOM root)")
print("QA / logs     :", QA_DIR, "|", LOG_DIR)
print("metadata      :", LOCAL_META, f"(bundle folder {META_DIRNAME!r} from config)",
      " flags from:", FLAGS_SOURCE)

## 2. GATE — the cloud copy of the DICOMs must be COMPLETE

**Read this before you run the next cell.** Getting the images to Colab is **two** transfers, not one:

1. **Globus** moves 6,122 DICOMs (~90 GB) from Nightingale into the Drive folder on the Mac. When
   Globus reports success, the files exist **on the local disk**.
2. **Google Drive for Desktop** then has to upload all ~90 GB **to the cloud**, and that is a
   second, much slower transfer that can run for many hours or days on a home connection. It shows
   no progress bar worth trusting.

**Colab only ever sees the cloud copy.** A file that Globus finished but Drive has not uploaded is
simply *absent* here. So a count below 6,122 in this cell does not mean the transfer failed — it
almost always means **the Drive upload is still in flight.**

> **If this gate fails, DO NOT PROCEED.** Preprocessing a partial set silently drops patients and
> views: the missing-view mask written into every shard would record views the patient actually has
> as missing, and that wrong mask is read by the training notebook. Wait for the Drive menu-bar icon
> to report "Sync complete", then re-run this cell with `FORCE_RESCAN = True`.

The gate checks **set equality of relative paths** against `image_transfer_manifest_paths.txt`, not
just the count — 6,122 files of which one is the wrong file would pass a count check.

Walking ~3,700 nested Drive folders takes a few minutes, so the scan is cached to local disk; set
`FORCE_RESCAN = True` in cell 0 to re-walk.

In [ ]:
# ---- 2. GATE: the Drive copy must contain exactly the 6,122 manifest DICOMs -------------------
EXPECTED_N   = int(_raw["transfer"]["expected_n_files"])
DICOM_SUFFIX = str(_raw["transfer"]["dicom_suffix"])
MIN_BYTES    = int(_raw["transfer"]["min_file_bytes"])
IGNORED_NAMES = {".DS_Store", "Icon\r", ".localized"}

manifest_rel, _seen = [], set()
for raw in MANIFEST_TXT.read_text().splitlines():        # same normalization as verify_transfer
    rel = raw.strip().lstrip("/")
    if rel and rel not in _seen:
        _seen.add(rel); manifest_rel.append(rel)
print(f"manifest: {len(manifest_rel):,} unique relative paths (config expects {EXPECTED_N:,})")
assert len(manifest_rel) == EXPECTED_N, "the bundled manifest does not match transfer.expected_n_files"

if CACHE_JSON.exists() and not FORCE_RESCAN:
    found = json.loads(CACHE_JSON.read_text())
    print(f"using cached Drive scan ({len(found):,} files) -- set FORCE_RESCAN=True to re-walk")
else:
    t0, found, others = time.time(), {}, []
    for dirpath, _dirs, files in os.walk(DICOM_ROOT):
        for name in files:
            if name in IGNORED_NAMES:
                continue
            full = Path(dirpath) / name
            rel = full.relative_to(DICOM_ROOT).as_posix()
            if name.lower().endswith(DICOM_SUFFIX.lower()):
                try:
                    found[rel] = full.stat().st_size
                except OSError:
                    found[rel] = -1
            else:
                others.append(rel)
    CACHE_JSON.write_text(json.dumps(found))
    print(f"walked the Drive DICOM tree in {time.time() - t0:.0f}s: {len(found):,} *{DICOM_SUFFIX}, "
          f"{len(others):,} other files")

missing   = [p for p in manifest_rel if p not in found]
extra     = sorted(p for p in found if p not in _seen)
too_small = [(p, found[p]) for p in manifest_rel if p in found and 0 <= found[p] < MIN_BYTES]

print(f"\npresent      : {len(manifest_rel) - len(missing):,} / {len(manifest_rel):,} "
      f"({100 * (len(manifest_rel) - len(missing)) / len(manifest_rel):.2f}%)")
print(f"missing      : {len(missing):,}")
print(f"extra .dcm   : {len(extra):,}")
print(f"below {MIN_BYTES:,} B: {len(too_small):,}")
if missing:
    print("\nfirst few missing (relative paths carry the de-identified patient id, so only 3 shown):")
    for p in missing[:3]:
        print("   ", p)

GATE_OK = (len(found) == EXPECTED_N and not missing and not extra and not too_small)
print("\nGATE:", "PASS -- the cloud copy is complete" if GATE_OK else "FAIL -- DO NOT PROCEED")
if not GATE_OK and missing and not extra:
    print("The usual cause is that Google Drive for Desktop has not finished uploading. Wait for "
          "'Sync complete' on the Mac, then re-run this cell with FORCE_RESCAN = True.")
assert GATE_OK, (
    f"Drive gate FAILED: {len(found):,} of {EXPECTED_N:,} DICOMs visible to Colab, {len(missing):,} "
    f"missing, {len(extra):,} unexpected, {len(too_small):,} truncated. Preprocessing a partial set "
    f"corrupts the per-patient missing-view mask. Fix the upload first.")

## 3. THE FROZEN PIPELINE — this cell must stay byte-identical to `src/preprocess_images.py`

**If you change this cell, change `src/preprocess_images.py` too, and vice versa.** Everything
below the `Config` shim is a verbatim slice of that module (lines 80-969 and 975-979); the shim is
`src/config.py` lines 24-55 with `PROJECT_ROOT` rebound to the staged metadata directory, which is
what lets `build_manifest` be copied without edits. Nothing here was re-derived by hand.

Port fingerprint (`sha256` of the ported region): `ac5fea9b4d2e9dd28d14bc1593df7d0c4595dfbc5f6d062840fab7b6b730cc3c`

### The contract this cell implements

**Half-select sign — the single worst failure mode.** `bilateral_display_convention: "radiological"`
means the patient's **RIGHT** side is displayed on the **IMAGE LEFT**. On a flip-corrected array:

| `contra_side` | half taken | columns |
| --- | --- | --- |
| `R` | **LEFT** | `arr[:, :mid - inset]` |
| `L` | **RIGHT** | `arr[:, mid + inset:]` |

with `mid = W // 2` and `inset = round(half_inset_frac * W)` moving the cut **away from the midline,
into the kept half**, so the midline strip is discarded rather than kept. This applies **only** when
`laterality == 'B'`. For `laterality in ('L','R')` the film is already single-knee, so
`laterality == contra_side` is asserted (`laterality_mismatch` otherwise) and the whole film is used.

**Normalization** (whole film, before half-select, per image only):
`apply_modality_lut` -> `apply_voi_lut` **only** when `WindowCenter`+`WindowWidth` or `VOILUTSequence`
is present -> invert if `PhotometricInterpretation == 'MONOCHROME1'` (`arr.max() - arr`) -> clip to
the 0.5 / 99.5 percentiles -> min-max to `[0, 1]`. **No cross-image or cross-split statistic exists**
(protocol section 12). Both LUT steps degrade rather than fail, and the degradation is counted, not
swallowed.

**Geometry.** `side = base_extent / (1 - 2 * crop_margin_frac)`, clamped to
`[min_crop_frac, max_crop_frac] * min(H, W)` of the (half-)image; out-of-bounds is **padded with the
median of a 2% border ring, never wrapped**; LANCZOS resize to 512; **then** `mask_borders`. Masking
runs *after* the resize deliberately — a LANCZOS kernel straddling the band boundary would otherwise
bleed a bright marker back inside the frame.

**`masked_pct` and the protocol section 13 exclusions.** `square_crop` returns `(crop, padded_frac)`;
`finalize_image` returns `(uint8, masked_pct)` where
`masked_pct = border_band_fraction(out_size, mask_border_frac) + padded_frac`, clipped to 1. At
`out_size 512` / `mask_border_frac 0.06` the band is `round(0.06 * 512) = 31` px per edge and the
fraction is **0.22752**. Exclusion precedence, evaluated in this order:

1. `masked_pct > max_masked_pct (0.35)` -> **`excessive_masking`**
2. else `exclude_failed_localization and (method == "fallback_center" or confidence < min_crop_confidence (0.10))` -> **`localization_failed`**

Both go to the failure report and **never into a shard**.

**Shards.** `{split}-{index:05d}.tar`, stdlib `tarfile`, <= `max_shard_mb` (400), **one writer per
split so no tar ever mixes splits**. Sample key `{empi_anon}_{view}_{sha1(sop_uid)[:12]}`,
dot-free because WebDataset splits a member name on its first dot. Members `{key}.png` then
`{key}.json`, written back-to-back so they are contiguous. `TarInfo.mtime = 0`, `mode = 0o644`.
PNG: 8-bit grayscale mode `"L"`, `optimize=False`, `compress_level=6`.

**One substitution, and only one:** the module's `assert_out_dir_is_outside_repo` (which refuses to
write identifier-bearing shards into the git repo) is not portable to Colab, where there is no repo.
Its Colab analogue is the assertion in cell 1 that `SHARD_DIR` is not inside `DICOM_ROOT`.

In [ ]:
# =================================================================================================
#    verbatim port of src/preprocess_images.py (lines 112-997, 1003-1007).
#    DO NOT EDIT THIS CELL IN ISOLATION. Change the module and re-port, or the crop QA gate that
#    was signed off locally does not apply to what this notebook writes.
# =================================================================================================
from __future__ import annotations

import hashlib
import io
import json
import logging
import os
import sys
import tarfile
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import scipy.ndimage as ndi
import yaml
from PIL import Image

# ---- src/config.py lines 24-55, verbatim; PROJECT_ROOT rebound to the staged bundle -----------
PROJECT_ROOT = LOCAL_META
DEFAULT_CONFIG = PROJECT_ROOT / "config" / "feasibility.yaml"


class Config(dict):
    """Thin dict wrapper with path-resolution helpers.

    Keeps the raw YAML structure accessible via normal dict access while adding
    a few convenience methods that resolve paths against PROJECT_ROOT.
    """

    def path(self, rel: str) -> Path:
        """Resolve a project-relative path to an absolute Path."""
        p = Path(rel)
        return p if p.is_absolute() else (PROJECT_ROOT / p)

    def source_path(self, key: str) -> Path:
        """Absolute path to a source CSV named in config['source_files'][key]."""
        fname = self["source_files"][key]["filename"]
        return self.path(self["paths"]["metadata_dir"]) / fname

    def parquet_path(self, key: str) -> Path:
        """Absolute path to the typed Parquet for a source table."""
        return self.path(self["paths"]["source_parquet_dir"]) / f"{key}.parquet"

    def out(self, rel_key: str) -> Path:
        """Absolute path for a configured output location, e.g. out('tables_dir')."""
        return self.path(self["paths"][rel_key])


def load_config(path: str | os.PathLike | None = None) -> Config:
    """Load feasibility.yaml and return a Config (dict subclass)."""
    cfg_path = Path(path) if path else DEFAULT_CONFIG
    with open(cfg_path, "r") as fh:
        data: dict[str, Any] = yaml.safe_load(fh)
    return Config(data)


# ---- src/preprocess_images.py lines 80-969, verbatim ------------------------------------------
MODULE = "preprocess_images"

# WebDataset splits a member name on its FIRST dot, so a sample key must not contain
# one. empi_anon is numeric, view is alphabetic, and the uid short is hex -> safe.
KEY_TEMPLATE = "{empi_anon}_{view}_{uid_short}"
UID_SHORT_LEN = 12
PNG_COMPRESS_LEVEL = 6          # PIL default; pinned so T5 reproduces identical bytes
LOCALIZER_WORK_PX = 384         # localization runs on a downsampled copy (speed only)

# Failure reasons raised by the protocol-section-13 exclusion rules.
REASON_EXCESSIVE_MASKING = "excessive_masking"
REASON_LOCALIZATION_FAILED = "localization_failed"


def setup_logging(log_path: Path) -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    lg = logging.getLogger(MODULE)
    lg.setLevel(logging.INFO)
    lg.propagate = False
    if not any(getattr(h, "_mrkr", False) for h in lg.handlers):
        fh = logging.FileHandler(log_path, mode="a")   # run.log is shared + APPEND-ONLY
        fh._mrkr = True  # type: ignore[attr-defined]
        fh.setFormatter(logging.Formatter(
            f"{MODULE} | %(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%dT%H:%M:%S"))
        lg.addHandler(fh)
        sh = logging.StreamHandler(sys.stdout)
        sh._mrkr = True  # type: ignore[attr-defined]
        sh.setFormatter(logging.Formatter(f"{MODULE} | %(levelname)s | %(message)s"))
        lg.addHandler(sh)
    return lg


# =============================================================================
# Parameters (every value comes from config `preprocess:` — nothing hard-coded)
# =============================================================================
@dataclass(frozen=True)
class PreprocessParams:
    """The frozen crop contract. T5's Colab notebook must construct this identically."""
    out_size: int
    apply_rescale: bool
    apply_voi_lut: bool
    monochrome1_invert: bool
    clip_percentiles: tuple[float, float]
    bilateral_display_convention: str
    apply_horizontal_flip_correction: bool
    half_inset_frac: float
    crop_margin_frac: float
    min_crop_frac: float
    max_crop_frac: float
    standardize_to_left: bool
    mask_border_frac: float
    localizer: str
    localizer_mode: str
    localizer_refine_min_confidence: float
    mask_markers: bool
    marker_sat_level: int
    marker_min_px: int
    marker_max_area_frac: float
    marker_ring_px: int
    marker_ring_max_level: int
    marker_merge_px: int
    image_format: str
    min_crop_confidence: float
    max_masked_pct: float
    exclude_failed_localization: bool

    @classmethod
    def from_config(cls, cfg) -> "PreprocessParams":
        p = cfg["preprocess"]
        lo, hi = p["clip_percentiles"]
        return cls(
            out_size=int(p["out_size"]),
            apply_rescale=bool(p["apply_rescale"]),
            apply_voi_lut=bool(p["apply_voi_lut"]),
            monochrome1_invert=bool(p["monochrome1_invert"]),
            clip_percentiles=(float(lo), float(hi)),
            bilateral_display_convention=str(p["bilateral_display_convention"]),
            apply_horizontal_flip_correction=bool(p["apply_horizontal_flip_correction"]),
            half_inset_frac=float(p["half_inset_frac"]),
            crop_margin_frac=float(p["crop_margin_frac"]),
            min_crop_frac=float(p["min_crop_frac"]),
            max_crop_frac=float(p["max_crop_frac"]),
            standardize_to_left=bool(p["standardize_to_left"]),
            mask_border_frac=float(p["mask_border_frac"]),
            localizer=str(p["localizer"]),
            localizer_mode=str(p.get("localizer_mode", "center_default")),
            localizer_refine_min_confidence=float(p.get("localizer_refine_min_confidence", 0.60)),
            mask_markers=bool(p.get("mask_markers", True)),
            marker_sat_level=int(p.get("marker_sat_level", 250)),
            marker_min_px=int(p.get("marker_min_px", 20)),
            marker_max_area_frac=float(p.get("marker_max_area_frac", 0.01)),
            marker_ring_px=int(p.get("marker_ring_px", 6)),
            marker_ring_max_level=int(p.get("marker_ring_max_level", 60)),
            marker_merge_px=int(p.get("marker_merge_px", 10)),
            image_format=str(p["shards"]["image_format"]),
            min_crop_confidence=float(p["min_crop_confidence"]),
            max_masked_pct=float(p["max_masked_pct"]),
            exclude_failed_localization=bool(p["exclude_failed_localization"]),
        )


@dataclass(frozen=True)
class Localization:
    """Output of any joint localizer. A learned detector in Colab must return THIS."""
    row: float
    col: float
    side: float
    confidence: float
    method: str


@dataclass(frozen=True)
class CropSpec:
    """Every geometric decision taken for one image (audited by src.crop_qa)."""
    view: str
    contra_side: str
    half_selected: str          # "left" | "right" | "none"
    orientation: str            # "left" | "right" — knee side as the crop READS
    mirrored: bool
    crop_method: str            # "intensity_profile" | "fallback_center"
    crop_confidence: float
    masked_pct: float           # protocol section 13: border band + out-of-bounds padding


class PreprocessError(RuntimeError):
    """Raised for a per-image failure that must be routed to the failure report."""

    def __init__(self, reason: str, detail: str = ""):
        super().__init__(f"{reason}: {detail}" if detail else reason)
        self.reason = reason
        self.detail = detail


# =============================================================================
# 1. DICOM decode
# =============================================================================
def _pixel_fns():
    """pydicom moved these helpers to pydicom.pixels in 3.x; support both locations."""
    try:
        from pydicom.pixels import apply_voi_lut, apply_modality_lut  # pydicom >= 3.0
    except Exception:                              # pragma: no cover - version fallback
        from pydicom.pixel_data_handlers.util import apply_voi_lut, apply_modality_lut
    return apply_voi_lut, apply_modality_lut


def read_dicom(path: str | Path, params: PreprocessParams) -> tuple[np.ndarray, dict]:
    """Decode one DICOM to a float32 2-D array in [0, 1] under the DISPLAY convention
    (higher value = brighter = denser bone).

    Order: pixel_array -> RescaleSlope/Intercept -> VOI LUT (window or LUT sequence)
    -> MONOCHROME1 inversion -> robust percentile clip -> min-max to [0, 1].

    The percentile clip is PER IMAGE. No statistic crosses image boundaries, so there
    is no cross-split calibration to leak (non-negotiable #2).

    Returns (array, meta) where meta carries PhotometricInterpretation, the
    `dicom_inverted` flag so the caller can cross-check the manifest `inverted` column,
    and `modality_lut_error` / `voi_lut_error`. Both LUT steps degrade rather than fail
    (a malformed LUT must not lose an otherwise usable image), but the degradation is
    NOT silent: the exception text is returned so the driver can count it and surface it
    in the run record instead of swallowing it.
    """
    import pydicom

    try:
        ds = pydicom.dcmread(str(path))
        arr = ds.pixel_array
    except Exception as exc:
        raise PreprocessError("decode_failed", f"{type(exc).__name__}: {exc}") from exc

    if arr.ndim == 3:
        arr = arr[arr.shape[0] // 2] if arr.shape[-1] not in (3, 4) else arr[..., 0]
    if arr.ndim != 2:
        raise PreprocessError("bad_shape", f"ndim={arr.ndim}")
    apply_voi, apply_modality = _pixel_fns()
    modality_lut_error = ""
    voi_lut_error = ""
    voi_lut_on_float = False

    # A VOI LUT *Sequence* is an INTEGER-INDEXED lookup table, so it should see the native
    # integer pixels; casting to float first makes pydicom warn "Applying a VOI LUT on a
    # float input array may give incorrect results".
    #
    # Scope, measured on real MRKR data (2026-07-25) — this is HARDENING, not a bug fix:
    # the modality transform is the IDENTITY in 100% of 300 sampled files (286 with
    # slope=1/intercept=0, 14 with no rescale tags, 0 ModalityLUTSequence) and 8% carry a
    # VOILUTSequence. Because an identity transform leaves the float values exactly integral,
    # the lookup landed on the same entries either way: old vs new output was bit-identical
    # on 8 VOILUTSequence images. Nothing was ever mis-decoded.
    #
    # It is kept because a non-integral RescaleSlope WOULD index the LUT wrongly and do so
    # silently. Staying integer when the transform is a no-op costs nothing; a genuinely
    # non-identity transform still forces float and is recorded in `voi_lut_on_float`.
    if params.apply_rescale and not _modality_is_identity(ds):
        try:
            arr = np.asarray(apply_modality(arr, ds), dtype=np.float32)
        except Exception as exc:
            modality_lut_error = f"{type(exc).__name__}: {exc}"
            slope = float(ds.get("RescaleSlope", 1.0) or 1.0)
            intercept = float(ds.get("RescaleIntercept", 0.0) or 0.0)
            arr = arr.astype(np.float32) * slope + intercept

    if params.apply_voi_lut:
        has_window = ds.get("WindowCenter", None) is not None and ds.get("WindowWidth", None) is not None
        has_lut = ds.get("VOILUTSequence", None) is not None
        if has_window or has_lut:
            voi_lut_on_float = bool(has_lut and np.issubdtype(arr.dtype, np.floating))
            try:
                arr = np.asarray(apply_voi(arr, ds), dtype=np.float32)
            except Exception as exc:
                # a malformed LUT must not lose the image; the rescaled pixels still train
                voi_lut_error = f"{type(exc).__name__}: {exc}"

    arr = np.asarray(arr, dtype=np.float32)

    photometric = str(getattr(ds, "PhotometricInterpretation", "") or "")
    dicom_inverted = photometric.upper() == "MONOCHROME1"
    if params.monochrome1_invert and dicom_inverted:
        arr = float(arr.max()) - arr

    arr = robust_scale(arr, params.clip_percentiles)
    meta = {
        "photometric": photometric,
        "dicom_inverted": bool(dicom_inverted),
        "rows": int(arr.shape[0]),
        "cols": int(arr.shape[1]),
        "modality_lut_error": modality_lut_error,
        "voi_lut_error": voi_lut_error,
        "voi_lut_on_float": bool(voi_lut_on_float),
    }
    return arr, meta


def _modality_is_identity(ds) -> bool:
    """True when the DICOM modality transform is a no-op, so the array can stay integer.

    Identity means: no ModalityLUTSequence, and RescaleSlope/Intercept are either absent or
    exactly (1, 0). Staying integer matters because a VOI LUT Sequence is indexed by pixel
    value (see read_dicom).
    """
    if ds.get("ModalityLUTSequence", None) is not None:
        return False
    slope = ds.get("RescaleSlope", None)
    intercept = ds.get("RescaleIntercept", None)
    slope_ok = slope is None or float(slope) == 1.0
    intercept_ok = intercept is None or float(intercept) == 0.0
    return bool(slope_ok and intercept_ok)


def robust_scale(arr: np.ndarray, clip_percentiles: tuple[float, float]) -> np.ndarray:
    """Per-image robust clip to [lo, hi] percentiles then min-max to [0, 1]."""
    lo_p, hi_p = clip_percentiles
    finite = np.isfinite(arr)
    if not finite.any():
        raise PreprocessError("degenerate_pixels", "no finite pixels")
    vals = arr[finite]
    lo = float(np.percentile(vals, lo_p))
    hi = float(np.percentile(vals, hi_p))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(vals.min()), float(vals.max())
    if hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    out = (np.clip(np.nan_to_num(arr, nan=lo), lo, hi) - lo) / (hi - lo)
    return out.astype(np.float32)


# =============================================================================
# 2-3. Flip correction and the CRITICAL half-select
# =============================================================================
def apply_flip_correction(arr: np.ndarray, horizontal_flip: int, enabled: bool) -> tuple[np.ndarray, bool]:
    """Mirror left-right when the manifest says the stored pixels are flipped.

    After this returns True the array obeys `bilateral_display_convention` exactly,
    so `select_contralateral_half` must be called with flip_corrected=True.
    """
    if enabled and int(horizontal_flip or 0) == 1:
        return np.ascontiguousarray(arr[:, ::-1]), True
    return arr, False


def contralateral_image_half(contra_side: str, horizontal_flip: int,
                             convention: str = "radiological",
                             flip_corrected: bool = True) -> str:
    """Which HALF of the pixel array (by COLUMN index) holds the CONTRALATERAL knee.

    Returns "left"  -> low  column indices, arr[:, :mid]
            "right" -> high column indices, arr[:, mid:]

    SIGN CONVENTION, spelled out because inverting it silently trains the model on the
    INDEX (already-replaced) knee, the single worst failure mode in this study:

      convention == "radiological"  (config bilateral_display_convention):
          the patient's RIGHT side is displayed on the IMAGE LEFT.
              contra_side == "R"  ->  "left"
              contra_side == "L"  ->  "right"
      convention == "anatomical":
          the patient's RIGHT side is displayed on the IMAGE RIGHT; both rows swap.

    `horizontal_flip` (MRKR image.horizontal_flip == 1) means the STORED pixels are
    mirrored left-right relative to the convention above.
          flip_corrected is True  -> the caller already un-mirrored the array via
                                     apply_flip_correction, so the flag must NOT be
                                     applied a second time.
          flip_corrected is False -> the array is raw, so horizontal_flip == 1 SWAPS
                                     the answer.
    Either call style yields the same anatomical knee; that redundancy is deliberate.
    """
    contra_side = str(contra_side).upper()
    assert contra_side in ("L", "R"), f"contra_side must be L or R, got {contra_side!r}"
    convention = str(convention).lower()
    assert convention in ("radiological", "anatomical"), f"unknown convention {convention!r}"

    half = "left" if contra_side == "R" else "right"          # radiological base case
    if convention == "anatomical":
        half = "right" if half == "left" else "left"
    if not flip_corrected and int(horizontal_flip or 0) == 1:
        half = "right" if half == "left" else "left"
    return half


def half_column_bounds(width: int, half: str, inset_frac: float) -> tuple[int, int]:
    """Column slice [c0, c1) of the selected half of a `width`-wide bilateral film.

    Factored out of `select_contralateral_half` so src.crop_qa can draw the EXACT
    rectangle the pipeline sliced onto the full film — the QA overlay is derived from
    the same arithmetic as the crop, never re-derived by eye.
    """
    assert half in ("left", "right"), f"half must be left|right, got {half!r}"
    w = int(width)
    mid = w // 2
    inset = int(round(float(inset_frac) * w))
    if half == "left":
        return 0, max(1, mid - inset)
    return min(w - 1, mid + inset), w


def select_contralateral_half(arr: np.ndarray, contra_side: str, horizontal_flip: int,
                              convention: str = "radiological",
                              flip_corrected: bool = True,
                              inset_frac: float = 0.0) -> tuple[np.ndarray, str]:
    """Slice the half of a BILATERAL frontal film that holds the contralateral knee.

    `inset_frac` (config half_inset_frac) is a fraction of the FULL image width and
    moves the cut AWAY from the midline, INTO the selected half, so the midline strip
    — where index-knee pixels could survive a slightly off-centre patient — is
    discarded rather than kept. Selecting the left half returns columns
    [0, mid - inset); the right half returns columns [mid + inset, W).

    Returns (half_array, half_name) with half_name in {"left", "right"}.
    """
    assert arr.ndim == 2, f"expected 2-D array, got ndim={arr.ndim}"
    half = contralateral_image_half(contra_side, horizontal_flip, convention, flip_corrected)
    c0, c1 = half_column_bounds(arr.shape[1], half, inset_frac)
    out = arr[:, c0:c1]
    assert out.shape[1] >= 1, "half-select produced an empty slice"
    return np.ascontiguousarray(out), half


# =============================================================================
# 4. Joint localization (classical, explainable; config localizer)
# =============================================================================
def otsu_threshold(img: np.ndarray, nbins: int = 256) -> float:
    """Otsu's threshold on a [0, 1] image. Implemented here so the module stays
    dependency-light (scikit-image is not installable on this interpreter).

    Returns the UPPER edge of the winning bin, not its centre: for a perfectly bimodal
    image the between-class variance is a plateau and argmax lands on the bin holding
    the background mode, so `img > centre` would keep the background as foreground.
    """
    hist, edges = np.histogram(np.clip(img, 0.0, 1.0), bins=nbins, range=(0.0, 1.0))
    hist = hist.astype(np.float64)
    total = hist.sum()
    if total <= 0:
        return 0.5
    centers = 0.5 * (edges[:-1] + edges[1:])
    p = hist / total
    omega = np.cumsum(p)                 # class-0 probability mass up to and including bin i
    mu = np.cumsum(p * centers)          # first-order cumulative moment
    mu_t = mu[-1]
    valid = (omega > 0.0) & (omega < 1.0)
    between = np.zeros(nbins, dtype=np.float64)
    between[valid] = ((mu_t * omega[valid] - mu[valid]) ** 2) / (omega[valid] * (1.0 - omega[valid]))
    return float(edges[int(np.argmax(between)) + 1])


def _fallback_localization(shape: tuple[int, int], params: PreprocessParams) -> Localization:
    """Deterministic centred crop. Sized at max_crop_frac: when localization fails a
    crop that is too SMALL can cut the joint out entirely, while one that is too large
    only adds context, and the half-select already guarantees no index-knee pixels."""
    h, w = shape
    short = float(min(h, w))
    return Localization(row=h / 2.0, col=w / 2.0, side=params.max_crop_frac * short,
                        confidence=0.0, method="fallback_center")


def mask_burned_in_markers(u8: np.ndarray, params: PreprocessParams) -> tuple[np.ndarray, int, float]:
    """Blank burned-in laterality letters / positioning markers left inside the crop.

    Protocol section 13 requires burned-in text to be masked, and non-negotiable #1 requires
    ZERO laterality markers to survive. A fixed `mask_border_frac` band only removes what sits
    at the very edge: measured on 500 real TRAIN crops, **26.4%** still carried a marker inside
    the band (frontal 15.8%, lateral 52.7%, sunrise 50.0%) — 'L', 'R', mirrored 'Я', technologist
    initials, arrows, circular anode markers.

    These are lead markers laid on the cassette, so they are SATURATED, SMALL, and sit on
    unexposed background. That triple signature is what separates them from bone:

      1. saturated:  value >= `marker_sat_level`
      2. small:      area within [`marker_min_px`, `marker_max_area_frac` x image]
      3. isolated:   the dilated ring around the blob is mostly dark
                     (median <= `marker_ring_max_level`), i.e. it sits on background,
                     whereas saturated bone is surrounded by mid-grey soft tissue

    The largest saturated component is never masked — that is bone or a collimator edge.
    Each masked blob is filled with its own ring median, not with 0, so the operation does not
    stamp a hard black rectangle that is itself a learnable artifact.

    Returns `(masked_u8, n_blobs_masked, fraction_of_pixels_masked)`.
    """
    if not params.mask_markers:
        return u8, 0, 0.0
    hot = u8 >= params.marker_sat_level
    if not hot.any():
        return u8, 0, 0.0
    lab, n = ndi.label(hot)
    if n == 0:
        return u8, 0, 0.0
    area = float(u8.size)
    sizes = ndi.sum(hot, lab, range(1, n + 1))
    keep_largest = int(np.argmax(sizes)) + 1          # bone / collimator edge — never mask
    candidates = hot & (lab != keep_largest)
    if not candidates.any():
        return u8, 0, 0.0

    # Text is MULTI-GLYPH: 'L', 'R', 'CROSS-TABLE', technologist initials are each several
    # separate components. Handling them one at a time strips some glyphs and leaves others,
    # which is worse than doing nothing — a half-erased marker is still a learnable artifact
    # and now carries a synthetic fill patch too. So dilate first to merge neighbouring
    # glyphs into one text REGION, then accept or reject the region as a whole.
    merged = ndi.binary_dilation(candidates, iterations=params.marker_merge_px)
    mlab, mn = ndi.label(merged)
    out = u8.copy()
    masked_px = 0
    n_masked = 0
    for k in range(1, mn + 1):
        region = mlab == k
        ink = float((candidates & region).sum())      # saturated pixels actually in the region
        if ink < params.marker_min_px or ink > params.marker_max_area_frac * area:
            continue
        ring = ndi.binary_dilation(region, iterations=params.marker_ring_px) & ~region
        if not ring.any():
            continue
        ring_med = float(np.median(out[ring]))
        if ring_med > params.marker_ring_max_level:
            continue                                   # sits on tissue, not background -> anatomy
        out[region] = np.uint8(round(ring_med))
        masked_px += int(region.sum())
        n_masked += 1
    return out, n_masked, masked_px / area


def center_localization(shape: tuple[int, int], params: PreprocessParams) -> Localization:
    """The PRIMARY crop centre: a deterministic centred box (method 'center_default').

    Geometrically identical to `_fallback_localization`, but it means the opposite thing.
    The fallback is "localization failed, salvage something"; this is "the centred box is
    the chosen estimate". Radiographers centre the knee in the collimated field, so the
    image centre is a strong, unbiased prior — measured on 800 real TRAIN films it beats
    the intensity-profile localizer, which locks onto bright shafts and collimator edges
    and drags the box off the joint (see config `preprocess.localizer_mode`).

    Confidence is 1.0 because this estimate is exact by construction: it is not a guess
    about anatomy, it is a deliberate policy. Nothing is excluded for using it.
    """
    h, w = shape
    short = float(min(h, w))
    return Localization(row=h / 2.0, col=w / 2.0, side=params.max_crop_frac * short,
                        confidence=1.0, method="center_default")


def choose_localization(img: np.ndarray, params: PreprocessParams) -> tuple[Localization, Localization]:
    """Pick the crop centre, and also return what the localizer thought, for auditing.

    Returns `(used, localizer_opinion)`. Under `localizer_mode == "center_default"` the
    centred box wins unless the localizer is BOTH non-fallback AND at least
    `localizer_refine_min_confidence` confident. Under "localizer_primary" the old
    behaviour is restored, so the change is reversible from config alone.
    """
    opinion = localize_joint(img, params)
    if params.localizer_mode == "localizer_primary":
        return opinion, opinion
    centre = center_localization(img.shape, params)
    refined = (opinion.method != "fallback_center"
               and float(opinion.confidence) >= params.localizer_refine_min_confidence)
    return (opinion if refined else centre), opinion


def localize_joint(img: np.ndarray, params: PreprocessParams) -> Localization:
    """Locate the tibiofemoral joint on an already half-selected single-knee image.

    Classical and explainable (config localizer == "intensity_profile"): bone is
    bright, so Otsu on a smoothed copy gives a bone mask; the largest connected
    component is the limb; its row-wise mass profile has two maxima (distal femur,
    proximal tibia) separated by the dark joint space, and the joint line is the
    minimum between them. The column centre is the mask centroid near that row.

    `crop_confidence` is the relative prominence of that valley, in [0, 1]. When the
    mask is degenerate or the valley is not prominent enough the function returns the
    deterministic centred fallback with method == "fallback_center" and confidence 0;
    the fallback rate is a headline QA number reported by src.crop_qa.

    A learned detector (Colab, task T5) may replace this body but MUST return the same
    Localization contract on the same input.
    """
    assert img.ndim == 2, f"expected 2-D array, got ndim={img.ndim}"
    h, w = img.shape
    short = float(min(h, w))
    lo_side = params.min_crop_frac * short
    hi_side = params.max_crop_frac * short
    if h < 16 or w < 16:
        return _fallback_localization((h, w), params)

    scale = min(1.0, LOCALIZER_WORK_PX / float(max(h, w)))
    small = ndi.zoom(img, scale, order=1) if scale < 1.0 else img
    sh, sw = small.shape
    if sh < 16 or sw < 16:
        return _fallback_localization((h, w), params)

    sm = ndi.gaussian_filter(small, sigma=max(1.0, min(sh, sw) / 64.0))
    mask = sm > otsu_threshold(sm)
    frac = float(mask.mean())
    if frac < 0.02 or frac > 0.90:          # no bone found, or the whole field is "bone"
        return _fallback_localization((h, w), params)

    lab, n = ndi.label(mask)
    if n == 0:
        return _fallback_localization((h, w), params)
    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    comp = lab == int(np.argmax(sizes))

    rows = np.flatnonzero(comp.any(axis=1))
    cols = np.flatnonzero(comp.any(axis=0))
    if rows.size < 8 or cols.size < 4:
        return _fallback_localization((h, w), params)
    r0, r1 = int(rows[0]), int(rows[-1])
    height = r1 - r0 + 1

    profile = ndi.gaussian_filter1d(comp.sum(axis=1).astype(np.float64),
                                    sigma=max(1.0, height / 50.0))
    # Search only the middle half of the limb: the taper at the top/bottom of the
    # field is not a joint space.
    b0 = r0 + int(round(0.25 * height))
    b1 = r0 + int(round(0.75 * height))
    if b1 - b0 < 3:
        return _fallback_localization((h, w), params)
    j = b0 + int(np.argmin(profile[b0:b1 + 1]))

    up = profile[r0:j]
    dn = profile[j + 1:r1 + 1]
    if up.size == 0 or dn.size == 0:
        return _fallback_localization((h, w), params)
    peak_up, peak_dn = float(up.max()), float(dn.max())
    row_up = r0 + int(np.argmax(up))
    row_dn = j + 1 + int(np.argmax(dn))
    peak = min(peak_up, peak_dn)
    if peak <= 0:
        return _fallback_localization((h, w), params)
    confidence = float(np.clip((peak - float(profile[j])) / peak, 0.0, 1.0))
    if confidence < params.min_crop_confidence:
        return _fallback_localization((h, w), params)

    band = max(2, int(round(0.10 * height)))
    near = comp[max(r0, j - band):min(r1 + 1, j + band + 1), :]
    near_cols = np.flatnonzero(near.any(axis=0))
    if near_cols.size >= 2:
        col_small = float(near_cols.mean())
        knee_width = float(near_cols[-1] - near_cols[0] + 1)
    else:
        col_small = float(cols.mean())
        knee_width = float(cols[-1] - cols[0] + 1)

    # crop_margin_frac is padding on EACH edge as a fraction of the crop side, so the
    # anatomy occupies (1 - 2 * margin) of the output.
    base = max(knee_width, float(row_dn - row_up))
    side_small = base / max(1e-6, 1.0 - 2.0 * params.crop_margin_frac)

    # ndi.zoom rounds the output shape, so map back with the ACTUAL axis ratios.
    inv_r, inv_c = float(h) / float(sh), float(w) / float(sw)
    side = float(np.clip(side_small * 0.5 * (inv_r + inv_c), lo_side, hi_side))
    return Localization(row=float(j) * inv_r, col=col_small * inv_c, side=side,
                        confidence=confidence, method="intensity_profile")


# =============================================================================
# 5-8. Crop, orientation, marker masking, resize
# =============================================================================
def border_value(arr: np.ndarray, ring_frac: float = 0.02) -> float:
    """The image's own edge/background level (median of a thin border ring)."""
    h, w = arr.shape
    t = max(1, int(round(ring_frac * min(h, w))))
    ring = np.concatenate([arr[:t, :].ravel(), arr[-t:, :].ravel(),
                           arr[:, :t].ravel(), arr[:, -t:].ravel()])
    return float(np.median(ring)) if ring.size else 0.0


def square_crop(arr: np.ndarray, center_rc: tuple[float, float], side: float,
                pad_value: float | None = None) -> tuple[np.ndarray, float]:
    """Square crop centred on center_rc, padded with the image's own background value
    where it runs off the array. Never wraps.

    Returns (crop, padded_frac) where `padded_frac` in [0, 1] is the fraction of the
    RETURNED crop that is invented padding rather than source pixels. Protocol section
    13 requires the masked-pixel percentage to be recorded and used for exclusion, and
    padding is masked pixels by another name: it carries no anatomy, only the film's own
    background level. `finalize_image` folds this into `masked_pct`.
    """
    assert arr.ndim == 2, f"expected 2-D array, got ndim={arr.ndim}"
    h, w = arr.shape
    n = max(2, int(round(float(side))))
    r, c = center_rc
    r0 = int(round(r - n / 2.0))
    c0 = int(round(c - n / 2.0))
    if pad_value is None:
        pad_value = border_value(arr)
    out = np.full((n, n), float(pad_value), dtype=np.float32)
    sr0, sc0 = max(0, r0), max(0, c0)
    sr1, sc1 = min(h, r0 + n), min(w, c0 + n)
    if sr1 > sr0 and sc1 > sc0:
        out[sr0 - r0:sr1 - r0, sc0 - c0:sc1 - c0] = arr[sr0:sr1, sc0:sc1]
        covered = float(sr1 - sr0) * float(sc1 - sc0)
    else:
        covered = 0.0
    padded_frac = float(np.clip(1.0 - covered / float(n * n), 0.0, 1.0))
    return out, padded_frac


def standardize_orientation(crop: np.ndarray, contra_side: str,
                            standardize_to_left: bool) -> tuple[np.ndarray, str, bool]:
    """Mirror right knees so every output reads as a LEFT knee.

    Returns (array, orientation, mirrored) with orientation in {"left", "right"}
    describing how the returned crop READS, not the patient's anatomy.
    """
    side = str(contra_side).upper()
    assert side in ("L", "R"), f"contra_side must be L or R, got {contra_side!r}"
    if standardize_to_left and side == "R":
        return np.ascontiguousarray(crop[:, ::-1]), "left", True
    return crop, ("left" if side == "L" else "right"), False


def mask_borders(img: np.ndarray, mask_border_frac: float, fill: float = 0.0) -> np.ndarray:
    """Blank `mask_border_frac` of every edge.

    Burned-in L/R markers and technique text sit at the film margins, so this is a
    blunt but fully auditable guard: it does not detect anything, it unconditionally
    destroys a fixed frame. The RESIDUAL risk — a marker projected inboard of the
    frame — is exactly what the src.crop_qa visual gate exists to catch.
    """
    out = img.copy()
    h, w = out.shape
    t_r = int(round(float(mask_border_frac) * h))
    t_c = int(round(float(mask_border_frac) * w))
    if t_r > 0:
        out[:t_r, :] = fill
        out[h - t_r:, :] = fill
    if t_c > 0:
        out[:, :t_c] = fill
        out[:, w - t_c:] = fill
    return out


def to_uint8(arr: np.ndarray) -> np.ndarray:
    """[0, 1] float -> uint8, deterministic rounding."""
    return np.clip(np.rint(np.clip(arr, 0.0, 1.0) * 255.0), 0, 255).astype(np.uint8)


def border_band_fraction(out_size: int, mask_border_frac: float) -> float:
    """Fraction of an out_size square destroyed by `mask_borders`.

    Uses the SAME integer band width mask_borders uses, so the number reported is the
    number actually blanked. At out_size 512 and mask_border_frac 0.06 the band is
    round(0.06 * 512) = 31 px per edge and the fraction is 1 - (450/512)^2 = 0.2276.
    """
    n = int(out_size)
    t = int(round(float(mask_border_frac) * n))
    inner = max(0, n - 2 * t)
    return float(np.clip(1.0 - (inner * inner) / float(n * n), 0.0, 1.0))


def finalize_image(crop: np.ndarray, out_size: int, mask_border_frac: float,
                   padded_frac: float = 0.0) -> tuple[np.ndarray, float]:
    """Resize the square crop to out_size, mask the border, and report `masked_pct`.

    Masking runs AFTER the resize (not in the numbered pipeline order) on purpose: a
    resize filter mixes masked and unmasked pixels at the band boundary, which can
    bleed a bright marker back INSIDE the frame. Masking last guarantees the band is
    exactly round(mask_border_frac * out_size) px of a constant value on every edge.

    PROTOCOL SECTION 13 — "Record the percentage of masked pixels and exclude crops with
    excessive masking or failed localization." The returned `masked_pct` (a FRACTION in
    [0, 1], stored in the sidecar column `masked_pct`) is

        border_band_fraction(out_size, mask_border_frac) + padded_frac

    i.e. the fixed border frame the pipeline destroys plus the out-of-bounds padding
    `square_crop` invented, clipped to 1.0. The sum is deliberately CONSERVATIVE: thin
    padding lies inside the border band and is therefore double-counted, so `masked_pct`
    is an upper bound on the truly-blank area and the exclusion rule errs toward
    dropping a marginal crop rather than training on one.
    """
    u8 = to_uint8(crop)
    img = Image.fromarray(u8, mode="L").resize((int(out_size), int(out_size)),
                                               Image.Resampling.LANCZOS)
    out = mask_borders(np.asarray(img), mask_border_frac, fill=0)
    masked_pct = float(np.clip(border_band_fraction(out_size, mask_border_frac)
                               + float(padded_frac), 0.0, 1.0))
    return out, masked_pct


def encode_image(u8: np.ndarray, image_format: str = "png") -> bytes:
    fmt = str(image_format).lower()
    assert fmt == "png", f"only lossless png is allowed, got {image_format!r}"
    buf = io.BytesIO()
    Image.fromarray(u8, mode="L").save(buf, format="PNG", optimize=False,
                                       compress_level=PNG_COMPRESS_LEVEL)
    return buf.getvalue()


# =============================================================================
# The whole per-image crop pipeline (pure: array in, array + CropSpec out)
# =============================================================================
def crop_stages(arr: np.ndarray, *, view: str, laterality: str, contra_side: str,
                horizontal_flip: int, params: PreprocessParams) -> dict:
    """The full crop pipeline, returning EVERY intermediate src.crop_qa needs to render.

    Steps: flip correction -> half-select (BILATERAL frontals only) -> joint
    localization -> square crop -> orientation standardization -> resize -> border mask
    -> protocol-section-13 masked-pixel accounting.

    For unilateral frontals and for lateral/sunrise films the whole image is already
    the contralateral knee, so half-select is skipped; `laterality == contra_side` is
    asserted instead and a violation raises PreprocessError("laterality_mismatch").

    Returns a dict:
        film          flip-CORRECTED full film, float [0,1], NOT mirrored — what
                      src.crop_qa panel A must show
        half_bounds   (c0, c1) columns of `film` that were kept, or None when the whole
                      film is the contralateral knee
        premirror     final uint8 crop BEFORE standardize_orientation mirrors it — what
                      src.crop_qa panel B must show, because the mirror is exactly what
                      destroys the reviewer's left/right cue
        image         the final uint8 crop written to the shard
        spec          CropSpec (includes masked_pct)
        reject_reason None, or the protocol-section-13 reason this crop must NOT be
                      written ("excessive_masking" | "localization_failed")

    Geometry errors are RETURNED, not raised, so the QA gate can still render a rejected
    crop and show a reviewer why it was dropped. `crop_contralateral_knee` raises them.
    """
    lat = str(laterality).upper()
    side = str(contra_side).upper()
    if side not in ("L", "R"):
        raise PreprocessError("contra_side_unresolved", side)

    arr, flip_applied = apply_flip_correction(arr, horizontal_flip,
                                              params.apply_horizontal_flip_correction)
    # The array now obeys the display convention iff it was never flipped OR we just
    # un-flipped it; otherwise select_contralateral_half must account for the flag itself.
    flip_corrected = flip_applied or int(horizontal_flip or 0) != 1
    if lat == "B":
        half, half_name = select_contralateral_half(
            arr, side, horizontal_flip,
            convention=params.bilateral_display_convention,
            flip_corrected=flip_corrected,
            inset_frac=params.half_inset_frac)
        half_bounds = half_column_bounds(arr.shape[1], half_name, params.half_inset_frac)
    elif lat in ("L", "R"):
        if lat != side:
            raise PreprocessError("laterality_mismatch", f"laterality={lat} contra_side={side}")
        half, half_name, half_bounds = arr, "none", None
    else:
        raise PreprocessError("laterality_unresolved", lat)

    loc, loc_opinion = choose_localization(half, params)
    crop, padded_frac = square_crop(half, (loc.row, loc.col), loc.side)
    premirror, masked_pct = finalize_image(crop, params.out_size, params.mask_border_frac, padded_frac)
    # Burned-in markers that survived the border band (protocol section 13, non-negotiable #1).
    # Marker pixels ARE masked pixels, so they count toward masked_pct exactly as the protocol
    # defines it ("record the percentage of masked pixels").
    premirror, n_markers_masked, marker_frac = mask_burned_in_markers(premirror, params)
    masked_pct = float(min(1.0, masked_pct + marker_frac))
    # Mirror the FINISHED premirror image rather than re-deriving the final crop from the
    # unmirrored array. Resize, border mask and marker masking all commute with a left-right
    # flip, so this is equivalent — but running the marker detector once instead of twice keeps
    # the two images EXACTLY equivariant (a component-size tie can otherwise resolve differently
    # on the mirrored copy) and halves the work. The QA sheet relies on that equivalence.
    _, orientation, mirrored = standardize_orientation(crop, side, params.standardize_to_left)
    out = np.ascontiguousarray(premirror[:, ::-1]) if mirrored else premirror.copy()

    # ---- protocol section 13 exclusions --------------------------------------------
    # Under localizer_mode == "center_default" a centred box is a deliberate choice, so it
    # is NOT a failure and exclude_failed_localization is off; only genuine masking excess
    # drops an image. The old behaviour remains reachable from config alone.
    localization_failed = (loc.method == "fallback_center"
                           or float(loc.confidence) < params.min_crop_confidence)
    reject_reason: str | None = None
    if masked_pct > params.max_masked_pct:
        reject_reason = REASON_EXCESSIVE_MASKING
    elif params.exclude_failed_localization and localization_failed:
        reject_reason = REASON_LOCALIZATION_FAILED

    spec = CropSpec(view=view, contra_side=side, half_selected=half_name,
                    orientation=orientation, mirrored=mirrored,
                    crop_method=loc.method, crop_confidence=round(float(loc.confidence), 4),
                    masked_pct=round(float(masked_pct), 4))
    assert out.shape == (params.out_size, params.out_size), f"bad output shape {out.shape}"
    return {"film": arr, "half_bounds": half_bounds, "half_selected": half_name,
            "localization": loc, "localizer_opinion": loc_opinion,
            "padded_frac": float(padded_frac), "n_markers_masked": int(n_markers_masked),
            "marker_frac": float(marker_frac),
            "premirror": premirror, "image": out, "spec": spec,
            "reject_reason": reject_reason}


def crop_contralateral_knee(arr: np.ndarray, *, view: str, laterality: str, contra_side: str,
                            horizontal_flip: int, params: PreprocessParams) -> tuple[np.ndarray, CropSpec]:
    """Decoded DICOM array -> final out_size x out_size uint8 contralateral-knee crop.

    Thin wrapper over `crop_stages` that enforces the protocol-section-13 exclusions:
    a crop whose masked fraction exceeds `max_masked_pct` raises
    PreprocessError("excessive_masking"), and (when config
    `exclude_failed_localization` is set) a crop whose joint localization fell back to
    the deterministic centred box, or whose confidence is below `min_crop_confidence`,
    raises PreprocessError("localization_failed"). Both land in the failure report; a
    FAILED localization must never reach a shard.
    """
    st = crop_stages(arr, view=view, laterality=laterality, contra_side=contra_side,
                     horizontal_flip=horizontal_flip, params=params)
    spec: CropSpec = st["spec"]
    if st["reject_reason"] == REASON_EXCESSIVE_MASKING:
        raise PreprocessError(REASON_EXCESSIVE_MASKING,
                              f"masked_pct={spec.masked_pct:.4f} > max_masked_pct="
                              f"{params.max_masked_pct:.4f} (border band "
                              f"{border_band_fraction(params.out_size, params.mask_border_frac):.4f}"
                              f" + padding {st['padded_frac']:.4f})")
    if st["reject_reason"] == REASON_LOCALIZATION_FAILED:
        raise PreprocessError(REASON_LOCALIZATION_FAILED,
                              f"method={spec.crop_method} confidence={spec.crop_confidence:.4f} "
                              f"< min_crop_confidence={params.min_crop_confidence:.4f}")
    return st["image"], spec


# =============================================================================
# 9. Shard writing (stdlib tarfile; WebDataset shards are plain tars)
# =============================================================================
def uid_short(sop_uid: str) -> str:
    """Stable, collision-resistant short id for a SOPInstanceUID_anon.

    sha1 of the UID string, first 12 hex chars. Reproducible in any language, safe to
    print in outputs/ (no empi_anon), and dot-free so it is a legal WebDataset key part.
    """
    return hashlib.sha1(str(sop_uid).encode("utf-8")).hexdigest()[:UID_SHORT_LEN]


def sample_key(empi_anon: str, view: str, sop_uid: str) -> str:
    """WebDataset sample key: {empi_anon}_{view}_{uid_short}. Must contain no '.'."""
    key = KEY_TEMPLATE.format(empi_anon=str(empi_anon), view=str(view), uid_short=uid_short(sop_uid))
    assert "." not in key, f"sample key must not contain '.': {key!r}"
    return key


def _tar_footprint(nbytes: int) -> int:
    """Bytes a member of size nbytes occupies in a tar (512 header + padded payload)."""
    return 512 + ((int(nbytes) + 511) // 512) * 512


class ShardWriter:
    """Writes WebDataset shards for ONE split. Never mixes splits in a tar.

    All members of a sample share the basename `key` and are written back-to-back, so
    they are CONTIGUOUS in the tar as WebDataset requires.
    """

    def __init__(self, out_dir: Path, split: str, name_pattern: str, max_shard_mb: float):
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.split = str(split)
        self.name_pattern = str(name_pattern)
        self.max_bytes = int(float(max_shard_mb) * 1024 * 1024)
        self.index = 0
        self.shards: list[str] = []
        self._tar: tarfile.TarFile | None = None
        self._bytes = 0
        self._n_in_shard = 0
        self.n_samples = 0

    @property
    def current_shard(self) -> str:
        return self.name_pattern.format(split=self.split, index=self.index)

    def _open(self) -> None:
        if self._tar is None:
            self._tar = tarfile.open(self.out_dir / self.current_shard, "w")
            self._bytes = 0
            self._n_in_shard = 0

    def _rotate(self) -> None:
        if self._tar is not None:
            self._tar.close()
            self.shards.append(self.current_shard)
            self._tar = None
            self.index += 1

    def write(self, key: str, members: list[tuple[str, bytes]]) -> str:
        """Write one sample. `members` is [(extension, payload), ...]; returns the shard name."""
        assert members, "a sample needs at least one member"
        size = sum(_tar_footprint(len(p)) for _, p in members)
        if self._tar is not None and self._n_in_shard > 0 and self._bytes + size > self.max_bytes:
            self._rotate()
        self._open()
        assert self._tar is not None
        for ext, payload in members:
            name = f"{key}.{ext}"
            info = tarfile.TarInfo(name=name)
            info.size = len(payload)
            info.mtime = 0                      # deterministic shards across re-runs
            info.mode = 0o644
            self._tar.addfile(info, io.BytesIO(payload))
        self._bytes += size
        self._n_in_shard += 1
        self.n_samples += 1
        return self.current_shard

    def close(self) -> None:
        if self._tar is not None:
            self._tar.close()
            self.shards.append(self.current_shard)
            self._tar = None


# =============================================================================
# Manifest assembly
# =============================================================================
# Regression anchors for build_manifest, VERIFIED against derived-data/cohort on
# 2026-07-24. The whole crop pipeline is downstream of this join, so a silent change in
# any input parquet (a re-run cohort build, a different split file, an upstream MRKR
# refresh) must fail HERE and loudly rather than quietly re-shape the training set.
MANIFEST_REGRESSION = {
    "n_rows_kept": 6090,            # 6,122 selected images - 32 'E'/'I' rows
    "n_dropped_views": 32,
    "n_patients": 3709,
    "n_frontal": 4269,
    "n_frontal_bilateral": 4075,    # laterality == 'B' -> needs the half-select
    "n_frontal_unilateral": 194,
    "n_lateral": 1659,
    "n_sunrise": 162,
    "n_horizontal_flip": 287,
    "n_inverted": 1476,
    "by_split": {"train": 4270, "val": 602, "test": 1218},
}


def build_manifest(cfg, log: logging.Logger | None = None) -> pd.DataFrame:
    """The image-grain work list: selected_study_images x final_cohort x patient_splits
    x image.parquet flags, restricted to config views_kept.

    Ends with the MANIFEST_REGRESSION assertions (see above) plus the two invariants the
    crop logic depends on: every kept NON-frontal row and every kept UNILATERAL frontal
    row must already have `laterality == contra_side`, because those images skip the
    half-select entirely and are cropped whole."""
    coh = cfg.path(cfg["paths"]["cohort_dir"])
    view_map = cfg["image"]["view_map"]
    views_kept = list(cfg["preprocess"]["views_kept"])

    img = pd.read_parquet(coh / "selected_study_images.parquet")
    fc = pd.read_parquet(coh / "final_cohort.parquet")[
        ["empi_anon", "StudyInstanceUID_anon", "index_side", "contra_side", "side_source",
         "n_concordant_signals", "event_indicator", "time_from_landmark", "view_set", "tier_name"]]
    sp = pd.read_parquet(coh / "patient_splits.parquet")[["empi_anon", "split"]]

    man = img.merge(fc, on=["empi_anon", "StudyInstanceUID_anon"], how="inner")
    man = man.merge(sp, on="empi_anon", how="left")
    assert man["split"].notna().all(), "some manifest images have no split assignment"

    flags = pd.read_parquet(cfg.parquet_path("image"),
                            columns=["SOPInstanceUID_anon", "horizontal_flip", "inverted"])
    man = man.merge(flags, on="SOPInstanceUID_anon", how="left")
    man["horizontal_flip"] = man["horizontal_flip"].fillna(0).astype(int)
    man["inverted"] = man["inverted"].fillna(0).astype(int)

    man["view"] = man["view_position"].map(view_map)
    n_all = len(man)
    man = man[man["view"].isin(views_kept)].reset_index(drop=True)
    if log is not None:
        log.info("manifest: %d image rows (%d patients); dropped %d rows outside views_kept=%s",
                 len(man), man["empi_anon"].nunique(), n_all - len(man), views_kept)
    assert man["SOPInstanceUID_anon"].is_unique, "duplicate SOPInstanceUID_anon in the manifest"

    _assert_manifest_regression(man, n_all, log)
    return man


def _assert_manifest_regression(man: pd.DataFrame, n_all: int,
                                log: logging.Logger | None = None) -> None:
    """Lock the verified manifest composition (see MANIFEST_REGRESSION)."""
    R = MANIFEST_REGRESSION
    vc = man["view"].value_counts().to_dict()
    frontal = man[man["view"] == "frontal"]
    bilat = frontal[frontal["laterality"] == "B"]
    uni_frontal = frontal[frontal["laterality"] != "B"]
    nonfrontal = man[man["view"] != "frontal"]
    by_split = man["split"].value_counts().to_dict()

    assert len(man) == R["n_rows_kept"], \
        f"manifest kept {len(man)} rows, expected {R['n_rows_kept']}"
    assert n_all - len(man) == R["n_dropped_views"], \
        f"dropped {n_all - len(man)} non-kept-view rows, expected {R['n_dropped_views']}"
    assert man["empi_anon"].nunique() == R["n_patients"], \
        f"manifest covers {man['empi_anon'].nunique()} patients, expected {R['n_patients']}"
    assert int(vc.get("frontal", 0)) == R["n_frontal"], f"frontal={vc.get('frontal')} != {R['n_frontal']}"
    assert int(vc.get("lateral", 0)) == R["n_lateral"], f"lateral={vc.get('lateral')} != {R['n_lateral']}"
    assert int(vc.get("sunrise", 0)) == R["n_sunrise"], f"sunrise={vc.get('sunrise')} != {R['n_sunrise']}"
    assert len(bilat) == R["n_frontal_bilateral"], \
        f"bilateral 'B' frontals={len(bilat)} != {R['n_frontal_bilateral']}"
    assert len(uni_frontal) == R["n_frontal_unilateral"], \
        f"unilateral frontals={len(uni_frontal)} != {R['n_frontal_unilateral']}"
    assert int((man["horizontal_flip"] == 1).sum()) == R["n_horizontal_flip"], \
        f"horizontal_flip==1 on {int((man['horizontal_flip'] == 1).sum())} != {R['n_horizontal_flip']}"
    assert int((man["inverted"] == 1).sum()) == R["n_inverted"], \
        f"inverted==1 on {int((man['inverted'] == 1).sum())} != {R['n_inverted']}"
    for split, n in R["by_split"].items():
        assert int(by_split.get(split, 0)) == n, \
            f"split {split} has {by_split.get(split, 0)} kept rows, expected {n}"

    # The half-select is applied ONLY to bilateral frontals. Everything else is cropped
    # whole, so its manifest laterality MUST already be the contralateral side — if that
    # ever stops holding, whole index-knee films would be written to the shards.
    bad_nf = int((nonfrontal["laterality"] != nonfrontal["contra_side"]).sum())
    bad_uf = int((uni_frontal["laterality"] != uni_frontal["contra_side"]).sum())
    assert bad_nf == 0, f"{bad_nf} kept non-frontal rows have laterality != contra_side"
    assert bad_uf == 0, f"{bad_uf} kept unilateral frontal rows have laterality != contra_side"
    if log is not None:
        log.info("manifest regression OK: %d rows / %d patients; frontal %d (B %d + uni %d), "
                 "lateral %d, sunrise %d; hflip==1 %d; splits %s",
                 len(man), man["empi_anon"].nunique(), R["n_frontal"], R["n_frontal_bilateral"],
                 R["n_frontal_unilateral"], R["n_lateral"], R["n_sunrise"],
                 R["n_horizontal_flip"], R["by_split"])


def view_masks(rows: pd.DataFrame, views_kept: list[str]) -> pd.DataFrame:
    """Per-patient missing-view mask: which of frontal/lateral/sunrise the patient has."""
    out = rows.groupby("empi_anon")["view"].agg(set).rename("views").reset_index()
    for v in views_kept:
        out[f"has_{v}"] = out["views"].apply(lambda s, v=v: v in s)
    out["n_views"] = out["views"].apply(len).astype(int)
    return out.drop(columns=["views"])


# =============================================================================
# Driver
# ---- src/preprocess_images.py lines 975-979, verbatim -----------------------------------------
def _fail_row(reason: str, row: pd.Series, detail: str = "") -> dict:
    """Failure records carry the sop-uid-short, NEVER empi_anon (outputs/ is not ignored)."""
    return {"reason": reason, "view": row.get("view", ""), "contra_side": row.get("contra_side", ""),
            "split": row.get("split", ""), "laterality": row.get("laterality", ""),
            "sop_uid_short": uid_short(row["SOPInstanceUID_anon"]), "detail": detail}


cfg = load_config(CFG_PATH)
params = PreprocessParams.from_config(cfg)
assert params.localizer == "intensity_profile", (
    f"cell 4 has not been enabled, so only the intensity_profile localizer is implemented here; "
    f"config says {params.localizer!r}")
assert params.out_size == 512 and params.image_format == "png"
log = setup_logging(LOG_DIR / "preprocess_colab.log")

shard_cfg      = cfg["preprocess"]["shards"]
views_kept     = list(cfg["preprocess"]["views_kept"])
sidecar_columns = list(cfg["preprocess"]["sidecar_columns"])
MAX_SHARD_MB   = float(MAX_SHARD_MB_OVERRIDE if MAX_SHARD_MB_OVERRIDE is not None
                       else shard_cfg["max_shard_mb"])
BAND = border_band_fraction(params.out_size, params.mask_border_frac)

print("params:", asdict(params))
print(f"\nborder band destroyed by mask_borders at {params.out_size} / {params.mask_border_frac}: "
      f"{BAND:.5f}  (cap max_masked_pct = {params.max_masked_pct})")
print("sidecar columns:", len(sidecar_columns))
print("shard pattern:", shard_cfg["name_pattern"], "| max_shard_mb:", MAX_SHARD_MB,
      "" if MAX_SHARD_MB_OVERRIDE is None else "  <-- OVERRIDDEN, see cell 6")
print("normalization: PER IMAGE robust clip at percentiles", list(params.clip_percentiles),
      "+ min-max to [0,1]. No statistic is pooled across images, so nothing can leak across splits.")

## 4. OPTIONAL — a learned knee-joint detector (DISABLED BY DEFAULT; leave it disabled)

The default localizer is the **same torch-free classical intensity-profile detector as the local
module**: Otsu on a smoothed copy gives a bone mask, the largest connected component is the limb,
its row-wise mass profile has two maxima (distal femur, proximal tibia) separated by the dark joint
space, and the joint line is the minimum between them. `crop_confidence` is the relative prominence
of that valley. It is deterministic, fully auditable, has no weights to download, and — the point —
produces **results identical to `python3 -m src.preprocess_images`**, so the crop QA that gets signed
off locally is evidence about the shards this notebook writes.

**Swapping in a learned detector breaks that.** If you enable the cell below:

- it must return the same `Localization(row, col, side, confidence, method)` contract on the same
  input, and honour `min_crop_confidence` (0.10) so a low-confidence detection is still routed to
  `localization_failed` and kept out of the shards;
- protocol section 12 requires it to be **fit or selected on TRAINING PATIENTS ONLY** — a detector
  tuned by looking at val or test crops is leakage, and "I only eyeballed a few" counts;
- **the existing crop QA sign-off is invalidated.** Different crops are different evidence. You must
  re-run cell 9, re-sample, and get a fresh reviewer sign-off before training;
- the shards will no longer match what the local module produces, so record which localizer wrote
  them (`preprocess_run.json` carries `params.localizer`).

Nothing here downloads an external model by default. `torch` is imported **inside** the guard, so
the default path never imports it.

In [ ]:
# ---- 4. OPTIONAL learned localizer -- DISABLED. Read the markdown above before flipping this. ---
USE_LEARNED_LOCALIZER = False        # <-- leave False unless you are also re-running the QA gate

if USE_LEARNED_LOCALIZER:
    import torch                     # imported ONLY inside the guard: the default path is torch-free

    raise NotImplementedError(
        "No learned knee-joint detector is wired in. To use one:\n"
        "  1. load weights that were fit/selected on TRAINING PATIENTS ONLY (protocol section 12);\n"
        "  2. implement learned_localize(img, params) -> Localization(row, col, side, confidence,\n"
        "     method='learned_<name>') in the SAME coordinate frame localize_joint uses: `img` is\n"
        "     the already half-selected single-knee float array, `row`/`col` are pixel coordinates\n"
        "     in it, `side` is the square crop side in pixels, `confidence` is in [0, 1];\n"
        "  3. clamp `side` to [min_crop_frac, max_crop_frac] * min(H, W) exactly as localize_joint\n"
        "     does, and return the deterministic fallback for a failure so min_crop_confidence\n"
        "     still routes it to `localization_failed`;\n"
        "  4. rebind: globals()['localize_joint'] = learned_localize;\n"
        "  5. RE-RUN THE CROP QA GATE (cell 9) and obtain a fresh reviewer sign-off. The existing\n"
        "     sign-off describes intensity_profile crops and does not transfer.")
else:
    print("localizer:", params.localizer, "-- classical, torch-free, identical to "
          "src/preprocess_images.py. torch is NOT imported on this path.")
    print("torch imported:", "torch" in sys.modules)

## 5. Manifest and work list

`build_manifest` (verbatim, cell 3) joins `selected_study_images` x `final_cohort` x
`patient_splits` x the `image` flags and keeps `preprocess.views_kept`. It ends in
`MANIFEST_REGRESSION` assertions, so a stale or re-run bundle **fails here, loudly**, instead of
quietly reshaping the training set:

```
6,122 selected images -> 6,090 kept after dropping 32 'E'/'I' rows
frontal 4,269 = 4,075 bilateral 'B' (half-select) + 194 unilateral
lateral 1,659 + sunrise 162          (already contralateral-side-only, no half-select)
horizontal_flip == 1 on 287; inverted == 1 on 1,476; 3,709 patients
kept rows by split: train 4,270 / val 602 / test 1,218
```

The gate in cell 2 counts **6,122** because that is what Globus moved; the work list is **6,090**
because 32 `other`-view images are dropped by `views_kept`. Both numbers are correct.

The per-patient **missing-view mask** is computed from the FULL manifest **before** `SPLITS` or
`LIMIT` narrow the work list. Deriving it from a truncated list would record `n_views = 1` for a
patient whose lateral and sunrise films simply were not scheduled in this run, and that wrong mask
would be written into the shard the model reads.

**`horizontal_flip` cannot be cross-checked.** It is a model-inferred MRKR annotation (dataset
caution B.4) with an unquantified error rate and no DICOM tag records it. It drives the half-select
on 287 bilateral films, and a false positive or negative there hands the model the **index** knee for
that image. There is no automated defence — that is exactly why cell 9 renders the full film with the
selected half outlined and why a human must sign it off. (`inverted`, by contrast, IS cross-checked
against `PhotometricInterpretation` on every decode; the DICOM tag wins.)

In [ ]:
# ---- 5. Build the manifest, the view mask, and the work list ----------------------------------
man = build_manifest(cfg, log)

# The mask is a property of the COHORT, not of this run: compute it BEFORE narrowing.
masks = view_masks(man, views_kept).set_index("empi_anon")

splits = list(SPLITS)
assert splits, "SPLITS is empty"
bad = [s for s in splits if s not in ("train", "val", "test")]
assert not bad, f"unknown split(s) {bad}; allowed: train, val, test"
if "test" in splits and not INCLUDE_TEST:
    raise SystemExit(
        "REFUSING to process the LOCKED test split without INCLUDE_TEST = True (success "
        "non-negotiable #4: the test set is read exactly once, after the model, ensemble rule, "
        "thresholds and analysis script are frozen).")
if "test" in splits:
    log.warning("*** SEALED TEST SPLIT UNLOCKED via INCLUDE_TEST -- this must happen ONCE, after "
                "the model/ensemble/thresholds/analysis script are frozen and the crop QA + "
                "laterality audit are signed off. ***")
    print("\n" + "!" * 96)
    print("!!  SEALED TEST SPLIT UNLOCKED. Non-negotiable #4 says this happens ONCE, after the")
    print("!!  model, ensemble rule, thresholds and analysis script are FROZEN and both QA gates")
    print("!!  are signed off. If any of that is not true, set INCLUDE_TEST = False and re-run.")
    print("!" * 96 + "\n")

work = man[man["split"].isin(splits)].sort_values(
    ["empi_anon", "view", "SOPInstanceUID_anon"], kind="mergesort").reset_index(drop=True)
n_scheduled_full = len(work)
if LIMIT is not None:
    work = work.head(int(LIMIT)).reset_index(drop=True)
    dropped = n_scheduled_full - len(work)
    log.warning("LIMIT %d: processing only the first %d of %d scheduled images; %d DROPPED. "
                "n_views in the sidecar remains the FULL-cohort mask; n_views_written will be "
                "smaller.", int(LIMIT), len(work), n_scheduled_full, dropped)
    print(f"*** LIMIT = {LIMIT}: {len(work):,} of {n_scheduled_full:,} scheduled images will be "
          f"processed; {dropped:,} are DROPPED. This is a smoke test, not a run. ***")

# The key is computable from the manifest alone, so a resumed run can skip an image WITHOUT
# opening it -- Drive FUSE I/O, not compute, is the bottleneck.
work["key"] = [sample_key(e, v, u) for e, v, u in
               zip(work["empi_anon"], work["view"], work["SOPInstanceUID_anon"])]
assert work["key"].is_unique, "sample keys are not unique"

log.info("scheduled %d images / %d patients for splits=%s (test %s)", len(work),
         work["empi_anon"].nunique(), splits, "INCLUDED" if "test" in splits else "SEALED")
print(f"scheduled {len(work):,} images / {work['empi_anon'].nunique():,} patients   splits={splits}"
      f"   test={'INCLUDED' if 'test' in splits else 'SEALED'}")
for s in splits:
    sub = work[work["split"] == s]
    print(f"  {s:<5} images={len(sub):>5}  patients={sub['empi_anon'].nunique():>5}  "
          f"views={sub['view'].value_counts().to_dict()}")
print(f"  bilateral 'B' frontals needing half-select: "
      f"{int(((work['view'] == 'frontal') & (work['laterality'] == 'B')).sum()):,}"
      f"   |  cropped whole: {int(((work['view'] != 'frontal') | (work['laterality'] != 'B')).sum()):,}")
print(f"  horizontal_flip==1: {int((work['horizontal_flip'] == 1).sum()):,}"
      f"   |  manifest inverted==1: {int((work['inverted'] == 1).sum()):,}")

if DRY_RUN:
    n_missing = sum(0 if (DICOM_ROOT / str(p)).exists() else 1 for p in work["dicom_path"])
    print(f"\nDRY RUN: no pixels read. {len(work) - n_missing:,}/{len(work):,} scheduled files "
          f"present under {DICOM_ROOT} ({n_missing:,} missing).")
    print(f"DRY RUN: would write shards named '{shard_cfg['name_pattern']}' (<= {MAX_SHARD_MB} MB) "
          f"with sidecar '{shard_cfg['sidecar_csv']}' into {SHARD_DIR}.")

## 6. Resumable shard writer and the per-image sample builder

A 6,090-image run over Drive FUSE is hours long and Colab sessions die. Three things make that
survivable without ever changing what a shard contains:

- **Shards are built on local disk and moved to Drive only when complete.** A killed session leaves
  a partial tar in ephemeral `/content`, never a corrupt tar on Drive. `ResumableShardWriter`
  subclasses the frozen `ShardWriter` and reimplements only `_rotate`/`close` to move the finished
  file and record it; the tar bytes it produces are byte-identical.
- **Resume state** lives in `shards/_progress/progress.json` (which shards are complete and which
  keys are in them) plus one `rows-{split}-{index}.jsonl` fragment per shard holding that shard's
  sidecar rows. `labels.csv` is rebuilt from the fragments at the end, so the sidecar can never
  describe samples that are not in a shard, or miss samples that are.
- **A resumed run skips a key without opening its DICOM.** The key is computed from the manifest.

An image that **failed** is not "done", so a resume re-attempts it. That is deliberate: a Drive
rate-limit error and a protocol section 13 exclusion look the same in the failure report, and the
first deserves another try. A genuine exclusion simply fails again, deterministically, and the
report dedupes on `sop_uid_short`.

`_progress/` holds `empi_anon` and full SOP UIDs, so it stays on Drive with the shards and never
goes near the repo.

**About `MAX_SHARD_MB_OVERRIDE`:** at ~200 KB per 512x512 PNG the config's 400 MB packs roughly
2,000 samples per shard, so `train` is about three shards and a resume can lose up to a shard's worth
of work. If your sessions keep getting cut short, set `MAX_SHARD_MB_OVERRIDE = 100` in cell 0. That
changes only how samples are **packed** into tars — never a sample's bytes, its key, or its sidecar
row — and the training notebook globs `{split}-*.tar`, so it is transparent downstream. It does mean
the shard *file list* differs from a local `src.preprocess_images` run.

`build_sample` is the per-image body of `src.preprocess_images.main()`'s try-block, factored into a
function so the equivalence test can call the exact code the loop calls.

In [ ]:
# ---- 6a. build_sample: EXACT mirror of the per-image try-block in src.preprocess_images.main() --
def build_sample(r, masks, params, dicom_root, views_kept):
    """One manifest row -> (key, members, record, meta). Raises PreprocessError on any per-image
    failure so the caller routes it to the failure report instead of aborting the run."""
    path = Path(dicom_root) / str(r["dicom_path"])
    if not path.exists():
        raise PreprocessError("file_not_found", "")
    arr, meta = read_dicom(path, params)
    out, spec = crop_contralateral_knee(
        arr, view=str(r["view"]), laterality=str(r["laterality"]),
        contra_side=str(r["contra_side"]), horizontal_flip=int(r["horizontal_flip"]),
        params=params)

    key = sample_key(r["empi_anon"], spec.view, r["SOPInstanceUID_anon"])
    m = masks.loc[r["empi_anon"]]
    record = {
        "empi_anon": str(r["empi_anon"]),
        "sop_uid": str(r["SOPInstanceUID_anon"]),
        "view": spec.view,
        "contra_side": spec.contra_side,
        "split": str(r["split"]),
        "event_indicator": int(r["event_indicator"]),
        "time_from_landmark": int(r["time_from_landmark"]),
        "shard": "",
        "key": key,
        "crop_method": spec.crop_method,
        "crop_confidence": float(spec.crop_confidence),
        "masked_pct": float(spec.masked_pct),
        "sop_uid_short": uid_short(r["SOPInstanceUID_anon"]),
        "index_side": str(r["index_side"]),
        "laterality": str(r["laterality"]),
        "horizontal_flip": int(r["horizontal_flip"]),
        "inverted": int(r["inverted"]),
        "half_selected": spec.half_selected,
        "orientation": spec.orientation,
        "mirrored": bool(spec.mirrored),
        "out_size": int(params.out_size),
        **{f"has_{v}": bool(m[f"has_{v}"]) for v in views_kept},
        "n_views": int(m["n_views"]),
    }
    payload = {k: v for k, v in record.items() if k != "shard"}
    members = [(params.image_format, encode_image(out, params.image_format)),
               ("json", json.dumps(payload, sort_keys=True).encode("utf-8"))]
    return key, members, record, meta


# ---- 6b. ResumableShardWriter: identical tar bytes, safer placement ----------------------------
class ResumableShardWriter(ShardWriter):
    """ShardWriter that builds each tar on local disk and moves it to Drive only when it is
    COMPLETE. Overrides only _rotate/close (placement + bookkeeping); the member bytes, their
    order, TarInfo.mtime=0 / mode=0o644 and the rotation arithmetic are the frozen base class."""

    def __init__(self, drive_dir, local_dir, split, name_pattern, max_shard_mb,
                 start_index=0, on_shard_done=None):
        super().__init__(local_dir, split, name_pattern, max_shard_mb)
        self.drive_dir = Path(drive_dir); self.drive_dir.mkdir(parents=True, exist_ok=True)
        self.index = int(start_index)
        self.on_shard_done = on_shard_done
        self._rows_in_shard: list[dict] = []

    def write_sample(self, key, members, record):
        shard = self.write(key, members)          # may rotate first; returns the shard written to
        record = dict(record); record["shard"] = shard
        self._rows_in_shard.append(record)
        return shard

    def _finish(self, name):
        src, dst = self.out_dir / name, self.drive_dir / name
        shutil.move(str(src), str(dst))
        rows, self._rows_in_shard = self._rows_in_shard, []
        if self.on_shard_done is not None:
            self.on_shard_done(self.split, name, rows, dst.stat().st_size)

    def _rotate(self):
        if self._tar is not None:
            name = self.current_shard
            self._tar.close(); self.shards.append(name); self._tar = None
            self._finish(name)
            self.index += 1

    def close(self):
        if self._tar is not None:
            name = self.current_shard
            self._tar.close(); self.shards.append(name); self._tar = None
            self._finish(name)


# ---- 6c. resume state --------------------------------------------------------------------------
PROGRESS_JSON = PROGRESS_DIR / "progress.json"
FAILURES_JSONL = PROGRESS_DIR / "failures.jsonl"


def load_progress():
    if not PROGRESS_JSON.exists():
        return {"version": 1, "splits": {}}
    try:
        return json.loads(PROGRESS_JSON.read_text())
    except Exception as exc:
        print(f"progress.json unreadable ({exc}); starting fresh (existing shards are NOT deleted)")
        return {"version": 1, "splits": {}}


def rows_path(split, name):
    return PROGRESS_DIR / f"rows-{Path(name).stem}.jsonl"


def verify_progress(prog, strict=False):
    """Drop any recorded shard whose tar is missing, resized, or (strict) whose member names do not
    match the recorded keys. Returns (done_keys, next_index_by_split, kept_shards_by_split)."""
    done, nxt, kept = set(), {}, {}
    for split, rec in prog.get("splits", {}).items():
        kept[split], hi = [], -1
        for sh in rec.get("shards", []):
            p = SHARD_DIR / sh["name"]
            ok = p.exists() and (sh.get("bytes") in (None, p.stat().st_size))
            if ok and strict:
                try:
                    with tarfile.open(p, "r") as tf:
                        names = {n.split(".", 1)[0] for n in tf.getnames()}
                    ok = names == set(sh["keys"])
                except Exception:
                    ok = False
            if not ok:
                print(f"  resume: DISCARDING {sh['name']} (missing/changed) -- its "
                      f"{len(sh['keys'])} samples will be reprocessed")
                continue
            kept[split].append(sh)
            done.update(sh["keys"])
            hi = max(hi, int(Path(sh["name"]).stem.split("-")[-1]))
        nxt[split] = hi + 1
    return done, nxt, kept


PROGRESS = load_progress()
if RESUME:
    DONE_KEYS, NEXT_INDEX, KEPT = verify_progress(PROGRESS, strict=RESUME_STRICT)
    PROGRESS = {"version": 1, "splits": {s: {"shards": KEPT.get(s, [])} for s in KEPT}}
    PROGRESS_JSON.write_text(json.dumps(PROGRESS))
else:
    DONE_KEYS, NEXT_INDEX = set(), {}
    stale = sorted(p.name for s in splits for p in SHARD_DIR.glob(f"{s}-*.tar"))
    if stale:
        print(f"*** RESUME = False and {len(stale)} shard(s) already exist for splits={splits}: "
              f"{', '.join(stale[:8])}{' ...' if len(stale) > 8 else ''}")
        print("*** They will be OVERWRITTEN or ORPHANED. A WebDataset glob picks orphans up "
              "alongside the new shards -- delete them before training if this run writes fewer.")

todo = work[~work["key"].isin(DONE_KEYS)].reset_index(drop=True)
print(f"\nresume: {len(DONE_KEYS):,} keys already in completed shards; "
      f"{len(todo):,} of {len(work):,} scheduled images to process")
for s in splits:
    print(f"  {s:<5} next shard index = {NEXT_INDEX.get(s, 0):05d}   "
          f"todo = {int((todo['split'] == s).sum()):,}")

## 7. Execute — one Drive read per DICOM, resumable, nothing aborts the run

- **Each DICOM is opened exactly once.** Drive FUSE I/O, not the CPU, is the bottleneck, and the
  Drive API will occasionally error mid-loop under sustained access. Already-done keys are skipped
  before the file is touched.
- **Every per-image failure is caught** and appended to the failure report — a malformed
  `time_from_landmark`, a decode error, a protocol section 13 exclusion — so one bad file cannot
  kill a run at image 5,000. A *transient* I/O error (Drive rate limit, `Input/output error`) is
  retried with backoff before it is recorded as a failure.
- **Progress bar with an ETA**, plus a periodic line into the Drive log so you can see how far a
  disconnected session got.

Expect roughly **2-6 images/second**, dominated by Drive read latency: about **30-60 minutes for
train+val (4,872 images)**. Keep the tab alive.

In [ ]:
# ---- 7. The run ---------------------------------------------------------------------------------
if DRY_RUN:
    raise SystemExit("DRY_RUN = True -- stopping before any pixel is read. Set DRY_RUN = False to run.")

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

TRANSIENT = ("OSError", "IOError", "TimeoutError", "ConnectionError", "Input/output error",
             "Transport endpoint", "Resource temporarily unavailable", "errno 5", "Errno 5")
N_RETRIES, RETRY_SLEEP_S = 3, 4.0


def _is_transient(detail: str) -> bool:
    return any(t.lower() in str(detail).lower() for t in TRANSIENT)


def _append_jsonl(path, obj):
    with open(path, "a") as fh:
        fh.write(json.dumps(obj) + "\n")


def on_shard_done(split, name, rows, nbytes):
    """Called the moment a shard lands on Drive: persist its sidecar rows, then its manifest entry.
    Rows FIRST, so a crash between the two loses a shard record but never leaves progress.json
    claiming rows that were never written."""
    with open(rows_path(split, name), "w") as fh:
        for rec in rows:
            fh.write(json.dumps(rec) + "\n")
    PROGRESS.setdefault("splits", {}).setdefault(split, {"shards": []})["shards"].append(
        {"name": name, "n": len(rows), "bytes": int(nbytes), "keys": [r["key"] for r in rows]})
    PROGRESS_JSON.write_text(json.dumps(PROGRESS))
    print(f"    -> {name} ({len(rows):,} samples, {nbytes / 1024 ** 2:.0f} MB) on Drive")


writers = {s: ResumableShardWriter(SHARD_DIR, LOCAL_STAGE / s, s, shard_cfg["name_pattern"],
                                   MAX_SHARD_MB, start_index=NEXT_INDEX.get(s, 0),
                                   on_shard_done=on_shard_done)
           for s in splits}

n_ok = n_fail = n_retry = n_inverted_disagree = n_modality_lut_error = n_voi_lut_error = 0
lut_error_examples: list[str] = []
t0 = time.time()
it = todo.itertuples(index=False)
bar = tqdm(it, total=len(todo), unit="img", smoothing=0.05) if tqdm is not None else it

for i, row in enumerate(bar):
    r = pd.Series(row._asdict())
    for attempt in range(N_RETRIES):
        try:
            key, members, record, meta = build_sample(r, masks, params, DICOM_ROOT, views_kept)
            assert key == str(r["key"]), "key computed from the manifest disagrees with build_sample"
            if bool(meta["dicom_inverted"]) != bool(int(r["inverted"]) == 1):
                n_inverted_disagree += 1
            if meta.get("modality_lut_error"):
                n_modality_lut_error += 1
                if len(lut_error_examples) < 5:
                    lut_error_examples.append(f"modality_lut: {meta['modality_lut_error']}")
            if meta.get("voi_lut_error"):
                n_voi_lut_error += 1
                if len(lut_error_examples) < 5:
                    lut_error_examples.append(f"voi_lut: {meta['voi_lut_error']}")
            writers[str(r["split"])].write_sample(key, members, record)
            n_ok += 1
            break
        except PreprocessError as exc:
            if _is_transient(exc.detail) and attempt < N_RETRIES - 1:
                n_retry += 1; time.sleep(RETRY_SLEEP_S * (attempt + 1)); continue
            _append_jsonl(FAILURES_JSONL, _fail_row(exc.reason, r, exc.detail)); n_fail += 1
            break
        except Exception as exc:                       # never let one bad file kill the run
            detail = f"{type(exc).__name__}: {exc}"
            if _is_transient(detail) and attempt < N_RETRIES - 1:
                n_retry += 1; time.sleep(RETRY_SLEEP_S * (attempt + 1)); continue
            _append_jsonl(FAILURES_JSONL, _fail_row("unexpected_error", r, detail)); n_fail += 1
            break

    if (i + 1) % 250 == 0:
        rate = (i + 1) / max(1e-6, time.time() - t0)
        eta = (len(todo) - i - 1) / max(1e-6, rate)
        msg = (f"{i + 1}/{len(todo)}  ok={n_ok} fail={n_fail} retry={n_retry}  "
               f"{rate:.1f} img/s  ETA {time.strftime('%H:%M:%S', time.gmtime(eta))}")
        log.info("  %s", msg)
        if tqdm is None:
            print("  " + msg)

for w in writers.values():
    w.close()

el = time.time() - t0
print(f"\nprocessed {n_ok:,} written / {n_fail:,} failed / {n_retry:,} transient retries "
      f"in {el / 60:.1f} min ({n_ok / max(1e-6, el):.1f} img/s)")
for s in splits:
    print(f"  {s:<5} shards this session: {writers[s].shards or 'none'}")

## 8. Finalize — `labels.csv`, `preprocess_run.json`, failure report

`labels.csv` is rebuilt from the per-shard row fragments, so it describes **exactly** the samples
that are in completed shards on Drive — no more, no less. Columns are `preprocess.sidecar_columns`
in config order (26 columns, `masked_pct` included), matching the module.

`n_views_written` is computed over the whole sidecar at the end, exactly as the module does. Splits
are patient-level, so a patient's rows never straddle two splits and the value is stable across
resumed sessions.

The failure report is written in two tiers, the same split the module uses: an **aggregate** CSV
(counts by reason x view x contra_side x split — no identifiers, safe to copy into the repo at
`outputs/tables/preprocess_failures.csv`) and a **detail** CSV keyed by `sop_uid_short` only, which
stays on Drive.

In [ ]:
# ---- 8. Rebuild the sidecar from the completed shards, write the run record --------------------
frag_rows = []
for frag in sorted(PROGRESS_DIR.glob("rows-*.jsonl")):
    shard_name = frag.stem.replace("rows-", "") + ".tar"
    if not (SHARD_DIR / shard_name).exists():
        print(f"  skipping {frag.name}: {shard_name} is not on Drive")
        continue
    frag_rows += [json.loads(ln) for ln in frag.read_text().splitlines() if ln.strip()]

assert frag_rows, "no samples were written -- see the failure report below"
side = pd.DataFrame(frag_rows).drop_duplicates(subset=["key"], keep="last")
# Row order: src.preprocess_images.main() appends rows in work order, i.e. sorted by
# (empi_anon, view, SOPInstanceUID_anon). Re-sorting on the same keys makes labels.csv
# byte-identical to a local run instead of merely equal as a set -- and it is order-stable
# across resumed sessions, which the fragment order is not.
side = side.sort_values(["empi_anon", "view", "sop_uid"], kind="mergesort").reset_index(drop=True)

ordered = [c for c in sidecar_columns if c in side.columns]
extras = [c for c in side.columns if c not in ordered]
side = side[ordered + extras]

written = view_masks(side, views_kept).set_index("empi_anon")
side["n_views_written"] = side["empi_anon"].map(written["n_views"]).astype(int)
drift = int((side["n_views"] != side["n_views_written"]).sum())
if drift:
    log.warning("view-mask drift on %d rows: a scheduled view was lost to a decode failure; group "
                "the sidecar by empi_anon for the authoritative mask", drift)
    print(f"*** view-mask drift on {drift:,} rows: a scheduled view is missing from the shards. "
          f"n_views (cohort truth) != n_views_written (what the model can actually see). The "
          f"training notebook MUST use n_views_written. ***")

SIDECAR = SHARD_DIR / str(shard_cfg["sidecar_csv"])
side.to_csv(SIDECAR, index=False)

# ---- failures ---------------------------------------------------------------------------------
fail_cols = ["reason", "view", "contra_side", "split", "laterality", "sop_uid_short", "detail"]
if FAILURES_JSONL.exists():
    fdf = pd.DataFrame([json.loads(ln) for ln in FAILURES_JSONL.read_text().splitlines() if ln.strip()],
                       columns=fail_cols)
    fdf = fdf[~fdf["sop_uid_short"].isin(side["sop_uid_short"])]        # later success wins
    fdf = fdf.drop_duplicates(subset=["sop_uid_short", "reason"], keep="last")
else:
    fdf = pd.DataFrame(columns=fail_cols)
agg = (fdf.groupby(["reason", "view", "contra_side", "split"], dropna=False).size()
          .rename("n_images").reset_index().sort_values(["n_images", "reason"], ascending=[False, True])
       if len(fdf) else pd.DataFrame(columns=["reason", "view", "contra_side", "split", "n_images"]))
agg.to_csv(SHARD_DIR / "preprocess_failures.csv", index=False)          # counts only, no identifiers
fdf.to_csv(PROGRESS_DIR / "preprocess_failures_detail.csv", index=False)  # sop_uid_short only
fail_counts = fdf["reason"].value_counts().to_dict() if len(fdf) else {}

# ---- run record --------------------------------------------------------------------------------
mp = side["masked_pct"].astype(float)
shards_by_split = {s: [sh["name"] for sh in PROGRESS.get("splits", {}).get(s, {}).get("shards", [])]
                   for s in sorted(PROGRESS.get("splits", {}))}
n_loc_failed = int(fail_counts.get(REASON_LOCALIZATION_FAILED, 0))
run_json = {
    "produced_by": "notebooks/preprocess_colab.ipynb",
    "out_dir": str(SHARD_DIR),
    "out_dir_local_macos": str(Path(str(_raw["transfer"]["dest_root"])).parent /
                               str(_raw["preprocess"]["shards"]["out_dir"])),
    "dicom_root": str(DICOM_ROOT),
    "splits": splits,                                   # requested in THIS session
    "splits_present": sorted(side["split"].unique().tolist()),   # everything labels.csv describes
    "include_test": bool(INCLUDE_TEST),
    "limit": int(LIMIT) if LIMIT is not None else None,
    "sidecar_csv": str(SIDECAR),
    "n_images_scheduled": int(len(work)),
    "n_images_written": int(len(side)),
    "n_patients_written": int(side["empi_anon"].nunique()),
    "n_failures": int(len(fdf)),
    "failures_by_reason": {str(k): int(v) for k, v in fail_counts.items()},
    "n_excluded_excessive_masking": int(fail_counts.get(REASON_EXCESSIVE_MASKING, 0)),
    "n_excluded_localization_failed": n_loc_failed,
    "max_masked_pct": float(params.max_masked_pct),
    "border_band_fraction": round(BAND, 4),
    "masked_pct": {"mean": round(float(mp.mean()), 4), "min": round(float(mp.min()), 4),
                   "p50": round(float(mp.quantile(0.50)), 4), "p90": round(float(mp.quantile(0.90)), 4),
                   "p99": round(float(mp.quantile(0.99)), 4), "max": round(float(mp.max()), 4),
                   "n_gt_border_only": int((mp > BAND + 1e-9).sum())},
    "fallback_rate": round(float((side["crop_method"] == "fallback_center").mean()), 4),
    "localization_failure_rate": round(n_loc_failed / max(1, len(side) + n_loc_failed), 4),
    "n_inverted_disagree": int(n_inverted_disagree),
    "pct_inverted_disagree": round(100.0 * n_inverted_disagree / max(1, len(side)), 3),
    "n_modality_lut_error": int(n_modality_lut_error),
    "n_voi_lut_error": int(n_voi_lut_error),
    "lut_error_examples": lut_error_examples,
    "n_transient_retries": int(n_retry),
    "shards": shards_by_split,
    "max_shard_mb": MAX_SHARD_MB,
    "max_shard_mb_overridden": MAX_SHARD_MB_OVERRIDE is not None,
    "params": {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(params).items()},
    "written_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
(SHARD_DIR / "preprocess_run.json").write_text(json.dumps(run_json, indent=2))

print(f"sidecar      -> {SIDECAR}  ({len(side):,} rows x {side.shape[1]} cols)")
print(f"run record   -> {SHARD_DIR / 'preprocess_run.json'}")
print(f"failures     -> {SHARD_DIR / 'preprocess_failures.csv'} (counts only) + "
      f"{PROGRESS_DIR / 'preprocess_failures_detail.csv'} (sop_uid_short only)")
print(f"\nsplits in labels.csv: {run_json['splits_present']}")
for s in run_json["splits_present"]:
    sub = side[side["split"] == s]
    print(f"  {s:<5} samples={len(sub):>5} patients={sub['empi_anon'].nunique():>5} "
          f"shards={len(shards_by_split.get(s, []))}")
print(f"\nhalf-select : {side['half_selected'].value_counts().to_dict()}")
print(f"orientation : {side['orientation'].value_counts().to_dict()}  "
      f"(standardize_to_left={params.standardize_to_left})")
print(f"masked_pct  : mean {mp.mean():.4f}  p50 {mp.quantile(0.50):.4f}  p90 {mp.quantile(0.90):.4f}  "
      f"max {mp.max():.4f}   (border band alone {BAND:.4f}, cap {params.max_masked_pct})")
print(f"protocol s13 EXCLUSIONS (never written): excessive_masking="
      f"{fail_counts.get(REASON_EXCESSIVE_MASKING, 0)}, localization_failed={n_loc_failed}")
print(f"PhotometricInterpretation vs manifest 'inverted' disagreements: {n_inverted_disagree:,}"
      f"/{len(side):,} ({100.0 * n_inverted_disagree / max(1, len(side)):.2f}%) -- the DICOM tag is "
      f"trusted, the manifest flag is reported only")
print(f"LUT fallbacks: modality {n_modality_lut_error}, voi {n_voi_lut_error}")
print(f"failures: {len(fdf):,} {fail_counts if len(fdf) else '{}'}")
print("\nCopy preprocess_run.json to derived-data/cohort/ on the Mac (and set \"out_dir\" to "
      f"\"{run_json['out_dir_local_macos']}\") so src/crop_qa.py can find the shards locally.")

## 9. CROP QA GATE — the evidence that decides whether training may start

**A contact sheet of finished crops is NOT a gate, and cannot be made into one.** On a pre-index
film **both knees are native** — the index TKA has not happened yet, so there is no prosthesis to
give the wrong half away. `standardize_to_left` then mirrors right knees, destroying the left/right
cue, and `mask_borders` blanks a 31-px frame that takes the burned-in L/R marker with it. A finished
crop is therefore **anatomically identical whether the correct or the wrong half was taken.** A
reviewer staring at 72 beautiful knee crops can tell you nothing at all about half-select.

So each tile pairs two panels, exactly like `src/crop_qa.py`:

- **Panel A — the full film, flip-corrected and NOT mirrored**, with the half the pipeline KEPT
  outlined in green and the discarded half labelled `INDEX (discarded)` in red. Judge half-select
  here and nowhere else. The rectangle is drawn from `half_column_bounds`, the same arithmetic the
  crop used, so it cannot disagree with what was sliced.
- **Panel B — the crop BEFORE the standardize-to-left mirror**, so the anatomy still reads with its
  true handedness and can be checked against panel A.

`draw_film_panel` / `draw_missing_film_panel` are ported verbatim from `src/crop_qa.py`, so a panel
rendered here is the panel the module renders.

**Reviewer must confirm all three** (config `crop_qa.signoff_criteria`):

1. the outlined half of the full film is the **CONTRALATERAL** knee (opposite `index_side`);
2. the crop contains **no pixels from the index knee or the midline**;
3. **no residual burned-in laterality marker or text**.

Sampling is `crop_qa.n_per_cell` (12) per `view x contra_side` cell from **train + val only** —
the test split stays sealed. Tiles carry an **opaque patient index**, never `empi_anon`. This
re-reads ~72 DICOMs, deliberately, as a second read of a small sample.

The sheet is written to Drive (it contains patient images) and **must not be committed**. Training
is BLOCKED until a reviewer signs it.

In [ ]:
# ---- 9. Two-panel crop QA evidence sheet -------------------------------------------------------
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

def _downsample(img: np.ndarray, max_px: int = 900) -> np.ndarray:
    step = max(1, int(np.ceil(max(img.shape) / float(max_px))))
    return img[::step, ::step]


def draw_film_panel(ax, stage: dict, label: str) -> None:
    """Panel A: the flip-corrected full film with the SELECTED half outlined."""
    film = _downsample(np.asarray(stage["film"], dtype=np.float32))
    h, w = film.shape
    ax.imshow(film, cmap="gray", vmin=0.0, vmax=1.0, aspect="auto")

    bounds = stage.get("half_bounds")
    scale = np.asarray(stage["film"]).shape[1] / float(max(1, w))
    if bounds is None:
        c0, c1 = 0, w
        kept_txt, other = "CONTRA (whole film)", None
    else:
        c0, c1 = int(bounds[0] / scale), int(bounds[1] / scale)
        kept_txt = "CONTRA (kept)"
        other = (c1, w) if c0 == 0 else (0, c0)
    ax.add_patch(mpatches.Rectangle((c0 - 0.5, -0.5), max(1, c1 - c0), h,
                                    fill=False, edgecolor="#00ff66", linewidth=1.6))
    ax.text((c0 + c1) / 2.0, h * 0.045, kept_txt, color="#00ff66", fontsize=4.0,
            ha="center", va="top", fontweight="bold")
    if other is not None:
        ax.text((other[0] + other[1]) / 2.0, h * 0.045, "INDEX\n(discarded)", color="#ff5555",
                fontsize=4.0, ha="center", va="top", fontweight="bold")
    ax.set_title(label, fontsize=4.2, pad=1.6)
    ax.set_xticks([]); ax.set_yticks([])


def draw_missing_film_panel(ax, why: str) -> None:
    ax.set_facecolor("0.15")
    ax.text(0.5, 0.5, "NO FULL FILM\n\nhalf-select NOT\nvisually verifiable\n\n" + why,
            ha="center", va="center", fontsize=4.2, color="#ff5555", fontweight="bold",
            transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])


qa_cfg = cfg["crop_qa"]
seed = int(cfg["reproducibility"]["random_seed"]) + int(QA_SEED_OFFSET)
qa_splits = [s for s in list(qa_cfg["splits_sampled"]) if s in set(side["split"])]
assert qa_splits, f"none of crop_qa.splits_sampled={list(qa_cfg['splits_sampled'])} is in the sidecar"
n_per_cell = int(qa_cfg["n_per_cell"])
pool = side[side["split"].isin(qa_splits)]

picks = []
for view in views_kept:
    for cs in ("L", "R"):
        cell = pool[(pool["view"] == view) & (pool["contra_side"] == cs)]
        if cell.empty:
            continue
        picks.append(cell.sample(n=min(n_per_cell, len(cell)), random_state=seed)
                         .assign(cell=f"{view}|{cs}"))
sampled = pd.concat(picks).reset_index(drop=True)

# opaque patient index: the sheet must never carry empi_anon
index_of = {e: f"P{i:03d}" for i, e in enumerate(sorted(sampled["empi_anon"].unique()))}
pd.DataFrame({"patient_index": list(index_of.values()), "empi_anon": list(index_of.keys())}).to_csv(
    PROGRESS_DIR / "crop_qa_index_key.csv", index=False)      # key stays with the patient data

paths = man.set_index("SOPInstanceUID_anon")
stages, n_stage_err = {}, 0
for r in sampled.itertuples(index=False):
    meta = paths.loc[str(r.sop_uid)]
    try:
        arr, _ = read_dicom(DICOM_ROOT / str(meta["dicom_path"]), params)
        st = crop_stages(arr, view=str(meta["view"]), laterality=str(meta["laterality"]),
                         contra_side=str(meta["contra_side"]),
                         horizontal_flip=int(meta["horizontal_flip"]), params=params)
        st["index_side"] = str(meta["index_side"]); st["laterality"] = str(meta["laterality"])
        st["horizontal_flip"] = int(meta["horizontal_flip"])
        stages[str(r.key)] = st
    except Exception as exc:
        n_stage_err += 1
        print(f"  QA re-read failed ({type(exc).__name__}: {exc})")

cells_qa = [c for c in (f"{v}|{cs}" for v in views_kept for cs in ("L", "R"))
            if (sampled["cell"] == c).any()]
n_rows, n_cols = max(1, len(cells_qa)), max(2, 2 * n_per_cell)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.5 * n_cols, 2.6 * n_rows), dpi=150, squeeze=False)
n_crop = n_film = 0
for i, cell in enumerate(cells_qa):
    sub = sampled[sampled["cell"] == cell].reset_index(drop=True)
    for j in range(n_per_cell):
        ax_a, ax_b = axes[i, 2 * j], axes[i, 2 * j + 1]
        for ax in (ax_a, ax_b):
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_linewidth(0.4)
        if j >= len(sub):
            ax_a.set_facecolor("0.92"); ax_b.set_facecolor("0.92"); continue
        r = sub.iloc[j]
        pidx = index_of.get(r["empi_anon"], "?")
        st = stages.get(r["key"])
        if st is not None:
            draw_film_panel(ax_a, st, f"{pidx} lat={st['laterality']} idx={st['index_side']} "
                                      f"con={r['contra_side']} hf={st['horizontal_flip']}\n"
                                      f"half={st['half_selected']}  ({r['view'][:4]})")
            n_film += 1
        else:
            draw_missing_film_panel(ax_a, f"{pidx} {r['view'][:4]} con={r['contra_side']}")
        if st is None:
            ax_b.set_facecolor("0.92")
            ax_b.text(0.5, 0.5, "missing", ha="center", va="center", fontsize=5); continue
        ax_b.imshow(st["premirror"], cmap="gray", vmin=0, vmax=255); n_crop += 1
        method = "ip" if r["crop_method"] == "intensity_profile" else "FB"
        ax_b.set_title(f"crop PRE-mirror\n{method} q={float(r['crop_confidence']):.2f} "
                       f"msk={100.0 * float(r['masked_pct']):.0f}%", fontsize=4.2, pad=1.6)
    axes[i, 0].set_ylabel(cell.replace("|", "\ncontra="), fontsize=6)

fig.suptitle("Contralateral-knee crop QA (Colab) -- LEFT panel: the full film, flip-corrected, with "
             "the half the pipeline KEPT outlined in green (the discarded INDEX half is marked red). "
             "RIGHT panel: that crop before the left/right mirror.\nBoth knees are native on every "
             "pre-index film, so the crop ALONE cannot tell you which half was taken -- judge "
             "half-select on the LEFT panel.", fontsize=7, y=0.997)
fig.tight_layout(rect=(0, 0, 1, 0.965))
QA_PNG = QA_DIR / "crop_qa_contact_sheet_colab.png"
fig.savefig(QA_PNG, bbox_inches="tight"); plt.close(fig)

print(f"QA sheet -> {QA_PNG}   ({n_film} film panels, {n_crop} crop panels, "
      f"{n_stage_err} re-read failures)")
print(f"cells: {cells_qa}")
print(f"sampled from splits {qa_splits} (test SEALED), seed {seed}")
print("\nREVIEWER MUST CONFIRM ALL THREE before training starts:")
for c in list(qa_cfg["signoff_criteria"]):
    print("  [ ]", c)
print("\nThis sheet contains patient images: it lives on Drive and must NOT be committed.")
try:
    from IPython.display import Image as _IPyImage, display
    display(_IPyImage(filename=str(QA_PNG)))
except Exception:
    pass

## 10. Shard integrity verification

Six checks, all of which must pass before `notebooks/train_colab.ipynb` reads a byte:

1. **Per-split sample counts match the manifest** — written + protocol section 13 exclusions +
   other failures accounts for every scheduled image, with nothing unexplained.
2. **`png` / `json` pair up** — every sample has exactly one of each, and no other extension exists.
3. **Members are contiguous per key** — WebDataset groups a sample by consecutive members sharing a
   basename; an interleaved tar silently yields half-samples.
4. **No tar mixes splits** — checked exhaustively from the sidecar (`shard -> split` must be
   single-valued) and confirmed against the tar's own `.json` payloads on a sample (or on every
   sample with `FULL_VERIFY = True`).
5. **Sidecar row count == shard sample count**, and the key sets are *equal* per shard, not merely
   the same size.
6. **Nothing is orphaned** — every `{split}-*.tar` on Drive is claimed by the sidecar, so a glob in
   the training notebook cannot pick up a leftover shard from a shorter earlier run.

In [ ]:
# ---- 10. Verify the shards ---------------------------------------------------------------------
# Cost note: this opens every shard and reads its member HEADERS (not payloads), so on Drive it is
# a few seconds per 400 MB tar, not a full download.
import random as _random
from collections import Counter

checks, sc_by_shard = [], {}
side_shards = sorted(side["shard"].unique())
disk_shards = sorted(p.name for p in SHARD_DIR.glob("*.tar"))

checks.append(("No orphan shards on Drive", set(disk_shards) == set(side_shards),
               f"{len(disk_shards)} tars on Drive, {len(side_shards)} claimed by labels.csv; "
               f"orphans={sorted(set(disk_shards) - set(side_shards))[:5]}"))

split_of_shard = side.groupby("shard")["split"].nunique()
checks.append(("No tar mixes splits (sidecar)", bool((split_of_shard == 1).all()),
               f"{int((split_of_shard > 1).sum())} shard(s) carry more than one split"))
bad_name = [sh for sh in side_shards
            if not sh.startswith(side.loc[side['shard'] == sh, 'split'].iloc[0] + "-")]
checks.append(("Shard name matches its split", not bad_name, f"{len(bad_name)} mismatched: {bad_name[:5]}"))

n_pairs = n_contig = n_extbad = 0
keyset_bad = []
for sh in side_shards:
    with tarfile.open(SHARD_DIR / sh, "r") as tf:
        names = tf.getnames()
    keys = [n.split(".", 1)[0] for n in names]
    exts = [n.split(".", 1)[1] if "." in n else "" for n in names]
    sc_by_shard[sh] = set(keys)
    n_extbad += sum(1 for e in exts if e not in (str(params.image_format), "json"))
    n_pairs += sum(1 for k, c in Counter(keys).items() if c == 2)
    runs, prev, seen = 0, None, set()
    for k in keys:                       # contiguity: a key must never reappear after another key
        if k != prev:
            if k in seen:
                runs += 1
            seen.add(k); prev = k
    n_contig += runs
    wanted = set(side.loc[side["shard"] == sh, "key"])
    if set(keys) != wanted:
        keyset_bad.append(f"{sh}: tar-only={len(set(keys) - wanted)}, "
                          f"sidecar-only={len(wanted - set(keys))}")

checks.append(("Per-shard key set == sidecar key set", not keyset_bad,
               "; ".join(keyset_bad[:4]) if keyset_bad else f"{len(side_shards)} shard(s) exact"))

checks.append(("Every sample is a png+json pair", n_pairs == len(side),
               f"{n_pairs:,} complete pairs vs {len(side):,} sidecar rows"))
checks.append(("Only png/json members", n_extbad == 0, f"{n_extbad} members with another extension"))
checks.append(("Members contiguous per key", n_contig == 0,
               f"{n_contig} key(s) split across non-adjacent runs"))
checks.append(("Sidecar rows == shard samples", sum(len(v) for v in sc_by_shard.values()) == len(side),
               f"{sum(len(v) for v in sc_by_shard.values()):,} tar samples vs {len(side):,} rows"))

# per-split accounting against the manifest
acct_ok, acct_detail = True, []
fail_by_split = fdf["split"].value_counts().to_dict() if len(fdf) else {}
for s in sorted(set(side["split"])):
    sched = int((work["split"] == s).sum())
    got, lost = int((side["split"] == s).sum()), int(fail_by_split.get(s, 0))
    ok = (got + lost == sched) if s in splits else True
    acct_ok &= ok
    acct_detail.append(f"{s}: written {got} + failed {lost} = {got + lost} vs scheduled {sched}"
                       f"{'' if ok else '  <-- UNEXPLAINED'}")
checks.append(("Per-split counts match the manifest", acct_ok, "; ".join(acct_detail)))

# confirm the tar's own json agrees with the sidecar
sample_keys = list(side["key"]) if FULL_VERIFY else _random.Random(0).sample(
    list(side["key"]), k=min(200, len(side)))
want = side.set_index("key")
n_json_bad = 0
for sh, grp in side[side["key"].isin(set(sample_keys))].groupby("shard"):
    wanted = {f"{k}.json": k for k in grp["key"]}
    with tarfile.open(SHARD_DIR / sh, "r") as tf:
        for m in tf:
            k = wanted.get(m.name)
            if k is None:
                continue
            pay = json.loads(tf.extractfile(m).read())
            row = want.loc[k]
            if (pay["split"] != row["split"] or pay["key"] != k
                    or pay["view"] != row["view"] or pay["empi_anon"] != row["empi_anon"]
                    or "shard" in pay):
                n_json_bad += 1
checks.append((f"Sample .json agrees with labels.csv ({'all' if FULL_VERIFY else len(sample_keys)})",
               n_json_bad == 0, f"{n_json_bad} mismatched payloads"))

print(f"{'CHECK':<48} {'RESULT':<6} DETAIL")
for name, ok, detail in checks:
    print(f"{name:<48} {'PASS' if ok else 'FAIL':<6} {detail}")
ALL_OK = all(ok for _, ok, _ in checks)
print("\nSHARD INTEGRITY:", "PASS" if ALL_OK else "FAIL")
print("Training is still BLOCKED until a reviewer signs the cell 9 QA sheet and the >=200-patient "
      "laterality audit passes.")
assert ALL_OK, "shard integrity FAILED -- do not train on these shards"

## HANDOFF TO T7 (`notebooks/train_colab.ipynb`)

Everything below is the **realized** contract — what this notebook actually writes.

### Where things are

```
/content/drive/MyDrive/Radiographic Prediction of Contralateral Knee Arthroplasty/
├── DICOMs-knee-imaging/          6,122 source DICOMs   (do NOT write anything in here)
├── colab-metadata/               the metadata bundle (config + 4 parquet + manifest paths)
├── shards/                       <-- READ THIS
│   ├── train-00000.tar …         WebDataset shards, {split}-{index:05d}.tar, <= 400 MB
│   ├── val-00000.tar …
│   ├── test-*.tar                ONLY IF the sealed test split was explicitly unlocked
│   ├── labels.csv                the sidecar, one row per written sample
│   ├── preprocess_run.json       the run record (see "which splits" below)
│   ├── preprocess_failures.csv   counts by reason (no identifiers)
│   └── _progress/                resume state + failure detail + the QA patient-index key
├── qa/crop_qa_contact_sheet_colab.png
└── checkpoints/                  (model_image.checkpoint_dir — T7 writes here)
```

Resolve these from config exactly as cell 1 does; do not hard-code them.

### WebDataset tuple

```python
import webdataset as wds
urls = sorted(glob(f"{SHARD_DIR}/train-*.tar"))          # never glob "*.tar": that mixes splits
ds = (wds.WebDataset(urls, shardshuffle=True)
        .decode("l")                                      # 8-bit grayscale, mode "L"
        .to_tuple("png", "json"))                         # preprocess.shards.image_ext == "png"
```

Each sample is **exactly two members**, `{key}.png` then `{key}.json`, contiguous.
`key = {empi_anon}_{view}_{sha1(SOPInstanceUID_anon)[:12]}` and contains no `.`.
The PNG is **512 x 512, 8-bit, single channel**, already contralateral-only, already mirrored so
every knee reads as a LEFT knee, already border-masked. **Do not flip it again** — a horizontal flip
augmentation would undo `standardize_to_left` and reintroduce the side cue the pipeline removed.
The outer 31 px on each edge are zero by construction; that is the `mask_borders` band
(`masked_pct` >= 0.22752 for every sample), not a bug.

### `labels.csv` — 26 columns, config order (`preprocess.sidecar_columns`)

| column | dtype | notes |
| --- | --- | --- |
| `empi_anon` | str | patient key; **group by this** for the missing-view mask and for any CV split |
| `sop_uid` | str | full `SOPInstanceUID_anon` |
| `view` | str | `frontal` \| `lateral` \| `sunrise` |
| `contra_side` | str | `L` \| `R` — the knee IN the crop |
| `split` | str | `train` \| `val` \| `test` — patient-level, locked |
| `event_indicator` | int | 0/1, contralateral TKA within the horizon |
| `time_from_landmark` | int | days from the day-90 landmark; the survival time |
| `shard` | str | the tar this sample is in |
| `key` | str | the WebDataset key; joins to the `json` member's `key` |
| `crop_method` | str | `intensity_profile` (`fallback_center` never reaches a shard) |
| `crop_confidence` | float | valley prominence in [0, 1] |
| `masked_pct` | float | border band + out-of-bounds padding; **>= 0.22752** always, capped at 0.35 |
| `sop_uid_short` | str | `sha1(sop_uid)[:12]` |
| `index_side` | str | the REPLACED knee — **never a predictor**, evaluation/audit only |
| `laterality` | str | `B` \| `L` \| `R` as filmed |
| `horizontal_flip` | int | MRKR flag, 0/1 |
| `inverted` | int | MRKR flag, 0/1 (the DICOM tag was authoritative during decode) |
| `half_selected` | str | `left` \| `right` \| `none` (whole film) |
| `orientation` | str | `left` for every sample while `standardize_to_left` is true |
| `mirrored` | bool | true when the crop was mirrored (i.e. `contra_side == "R"`) |
| `out_size` | int | 512 |
| `has_frontal` / `has_lateral` / `has_sunrise` | bool | **cohort** truth from the full manifest |
| `n_views` | int | 1-3, cohort truth |
| `n_views_written` | int | 1-3, **what actually reached a shard** |

The per-sample `.json` member is this row **minus `shard` and `n_views_written`**, serialized with
`json.dumps(..., sort_keys=True)`. It is enough to build a batch without touching `labels.csv`, but
`n_views_written` is only in the CSV.

### Missing-view mask rule (read this twice)

The multi-view aggregator needs to know which views a patient has. Use **`n_views_written` and the
per-patient set of `view` values obtained by grouping `labels.csv` on `empi_anon`** — not
`has_frontal` / `has_lateral` / `has_sunrise` / `n_views`, which describe the **cohort** and will
overstate availability whenever a scheduled image was excluded by the protocol section 13 rules or
lost to a decode failure. Cell 8 prints a loud warning when they diverge. In code:

```python
lab = pd.read_csv(SHARD_DIR / "labels.csv")
avail = lab.groupby("empi_anon")["view"].agg(set)        # authoritative per-patient view set
mask  = {p: [v in s for v in ["frontal", "lateral", "sunrise"]] for p, s in avail.items()}
assert (lab.groupby("empi_anon")["n_views_written"].nunique() == 1).all()
```

A patient can contribute **more than one image of the same view** (a repeated frontal in the same
study). Decide explicitly whether to pool them or pick one — `sop_uid`/`key` distinguishes them.

### Which splits were processed

`preprocess_run.json` records both:

- `"splits"` — what the operator requested in the **final session**;
- `"splits_present"` — every split `labels.csv` actually describes (all sessions, since it is
  rebuilt from the on-Drive shard fragments), plus `"include_test"`, `"limit"`, and the shard file
  list per split under `"shards"`.

**Trust `splits_present` and the sidecar's own `split` column**, and glob per split
(`train-*.tar`), never `*.tar`. If `test-*.tar` exists at all, the seal was broken deliberately;
verify why before reading it.

### Before training starts

1. `python3 -m src.verify_transfer` -> PASS (already required before this notebook ran).
2. The cell 9 QA sheet is **signed off** by a reviewer against all three criteria.
3. The **>=200-patient laterality audit** (protocol section 7 / 23) has passed.
4. Cell 10 shard integrity -> PASS.
5. The clinical baseline M0 exists, so the image model has a comparator to beat.